"""
Forecasting municipal fiscal variables for 2024: ARIMA baseline vs.
Single-Task (STL) and Multi-Task (MTL) entity-embedded LSTMs, each run with
two feature sets:
    "full"     -> 4 fiscal targets + 7 external drivers
    "internal" -> the 4 fiscal targets only, i.e. forecasts based purely on
                  each variable's own autocorrelation, with no external
                  context. This isolates how much the external drivers are
                  actually contributing.

That gives 5 models compared per target: ARIMA, MTL-full, STL-full,
MTL-internal, STL-internal.

HOW THE MTL/STL MODELS USE EXTERNAL VARIABLES
----------------------------------------------
- Externals are INPUT-only, never forecast. Both MTL and STL only ever have
  four output heads -- one per TARGET_VAR (see build_lstm_model). The 7
  EXTERNAL_VARS never appear as an output; nothing in this script forecasts
  next year's population, GDP, etc. They are only ever consumed as features
  describing the recent past.
- Entity embedding: a city's integer id is mapped through a small learned
  Embedding (EMBEDDING_DIM=4). Concatenating that vector with the LSTM's
  output and passing both through a Dense layer (see build_lstm_model) lets
  the network learn a different effective *slope* per city for how the
  lagged targets/externals map to next year's targets -- not just a
  different constant offset. This is what "entity-embedded" means: one
  shared set of LSTM weights, but city-specific behavior via the embedding.
- MTL vs STL differ only in how many heads sit on top of that shared
  backbone: MTL trains one network with four heads (out_cash_ga, out_una_ga,
  ...) whose gradients all flow back into the same shared_hidden layer
  ("hard parameter sharing" -- a target that's easy to learn can help the
  others). STL trains four completely separate networks (same architecture,
  independent weights), one per target.
- No 2024 external data is used. For window_size=3, predicting a city's
  2024 targets uses that city's *own* 2021, 2022 and 2023 values (targets
  and, for the "full" feature set, externals) as the LSTM input window
  (build_sequences: y at index i+window_size is predicted from features at
  indices [i, i+window_size)). Since the window always ends the year before
  the one being predicted, the 2024 rows in the external-variable columns
  are simply never read for prediction -- this is a genuine one-step-ahead
  forecast, not a fit using same-year externals.

HOW MISSING VALUES ARE HANDLED -- NO IMPUTATION, COMPLETE-CASE CITIES ONLY
----------------------------------------------------------------------------
Earlier drafts filled gaps with forward/back-fill and a cross-sectional
mean. That's gone: nothing in this script invents, interpolates, or
averages a value for a city that doesn't have one. Instead:
- `eligible_cities(df, cols)` returns only the cities that have a complete,
  non-missing value in every column in `cols`, for every year on record.
  A single missing cell anywhere in that city's history drops the WHOLE
  city from any model that needs those columns.
- "internal" models (ARIMA, MTL/STL-internal) only need TARGET_VARS
  complete, so they use `internal_cities`.
- "full" models (MTL/STL-full) need TARGET_VARS + EXTERNAL_VARS complete,
  so they use `full_cities`, a subset of `internal_cities` (a city can be
  clean on targets but still be dropped here for a gap in one external
  driver).
- Because eligibility differs by feature set, the different models are NOT
  all scored on the same cities. `main()` prints how many cities qualify
  for each set, and every row of the results table records `n_cities_used`
  so this is explicit rather than hidden in an average.
"""


In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Flatten, Concatenate
from tensorflow.keras.callbacks import EarlyStopping
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")

In [7]:
# make results repeatable
np.random.seed(42)
tf.random.set_seed(42)

In [8]:
# ============================================================== CONFIG ====
DATA_PATH = "variable.xlsx"
TARGET_VARS = ["opr_ratio_gn", "opr_ratio_ep", "cash_ratio_totasst","totdebt_to_asst","capital_to_asst", "funded_ratio_total"]
EXTERNAL_VARS = ["ln_pop_city", "ln_curgdp", "ln_psnl_incm", "employment", "ln_med_homevalue",
                    "property_rel","intg_rev_rel",
                    "disaster_event", "flood_dmg_tot",
                    "under_18%", "age_65_plus%", "white%", "bachelors%", "disability%", "poverty%"
                    ]
ENTITY_COL, YEAR_COL = "pid", "year"

WINDOW_SIZE = 3           # years of history the LSTM sees before predicting
TRAIN_END_YEAR = 2023     # last year usable as a TRAINING target
FORECAST_YEAR = 2024      # the held-out year to score against
# Changing the holdout year (e.g. TRAIN_END_YEAR=2022, FORECAST_YEAR=2023 to
# roll the test year back) is a pure config change -- nothing else to touch.
# NOTE -- this is still a ONE-STEP-AHEAD design: whatever FORECAST_YEAR you
# pick, the model's input window is the WINDOW_SIZE actual years immediately
# before it. Setting FORECAST_YEAR more than 1 year past TRAIN_END_YEAR does
# NOT give a true multi-year-ahead forecast -- it would use real, observed
# intervening-year data as input (if present in the file) rather than a
# genuine H-step projection. A real multi-step-ahead horizon needs a
# different (recursive) forecasting setup -- ask if you want that added.

# Validation split for early stopping. Without this, EarlyStopping watches
# TRAINING loss, which never tells you whether the model is overfitting --
# it only tells you when training has stopped improving on data it has
# already memorized. With VAL_YEARS > 0, the most recent VAL_YEARS training
# years are held out from gradient updates and used purely to decide when
# to stop, so "best weights" means best-on-unseen-years, not best-fit.
USE_VALIDATION = True
VAL_YEARS = 1             # e.g. 1 -> the single most recent training year

EMBEDDING_DIM, LSTM_UNITS, SHARED_DENSE_UNITS = 4, 16, 16
EPOCHS, BATCH_SIZE, PATIENCE = 200, 32, 15
# All five knobs above are free to change independently and are the ones
# worth sweeping first: WINDOW_SIZE (more/less history per prediction),
# LSTM_UNITS / SHARED_DENSE_UNITS / EMBEDDING_DIM (model capacity -- bigger
# isn't automatically better with only a few thousand training sequences),
# EPOCHS/PATIENCE (how long training is allowed to keep looking for
# improvement). None of them require touching any code below this point.

FEATURE_SETS = {
    "full": TARGET_VARS + EXTERNAL_VARS,
    "internal": TARGET_VARS,
}

In [9]:
# =========================================================== DATA PREP ====

def load_data(path):
    """Read Excel and keep only the needed columns. No filling/imputation
    happens here or anywhere else in this script -- missing values are
    handled purely by dropping incomplete cities (see eligible_cities)."""
    df = pd.read_excel(path)
    keep = [ENTITY_COL, "city", YEAR_COL] + TARGET_VARS + EXTERNAL_VARS
    return df[keep].sort_values([ENTITY_COL, YEAR_COL]).reset_index(drop=True)


def eligible_cities(df, cols):
    """Cities with NO missing value in any of `cols`, across every year
    they appear. Returns a sorted list of city ids (original, un-recoded)."""
    incomplete = df.loc[df[cols].isnull().any(axis=1), ENTITY_COL].unique()
    return sorted(set(df[ENTITY_COL].unique()) - set(incomplete))


def build_sequences(df, feature_cols, window_size):
    """Slide a window of `window_size` years per city.
    Returns X_seq (n, window, n_features), X_ent (n,), y (n, 4 targets),
    target_year (n,) -- the calendar year each row predicts."""
    X_seq, X_ent, y, target_year = [], [], [], []
    for pid, g in df.groupby(ENTITY_COL):
        g = g.sort_values(YEAR_COL)
        feats = g[feature_cols].values
        targs = g[TARGET_VARS].values
        years = g[YEAR_COL].values
        for i in range(len(g) - window_size):
            X_seq.append(feats[i:i + window_size])
            X_ent.append(pid)
            y.append(targs[i + window_size])
            target_year.append(years[i + window_size])
    return (np.array(X_seq, dtype="float32"), np.array(X_ent, dtype="int32"),
            np.array(y, dtype="float32"), np.array(target_year))


In [10]:
# ======================================================= CITY SCALING =====
# MIN-MAX SCALING -- summary
# - Each variable (every target and every external driver) gets its OWN
#   min/max -- never mixed with any other variable.
# - Each city gets its OWN min/max per variable -- never mixed with any
#   other city's history.
# - Parameters are estimated ONLY from that city's 2013-2023 (training)
#   years, so no 2024 information leaks into scaling.
# - Inputs (X) and targets (y) are scaled with separate scaler objects,
#   even for variables that appear in both roles.
# - A city's actual 2024 value can fall outside [0, 1] after scaling if
#   2024 was a new high/low relative to 2013-2023 -- expected, not a bug.
# - Predictions are inverse-transformed with that SAME city's parameters
#   before computing MAE / sMAPE, so errors are reported in original units.
#
# One MinMaxScaler per city, fit on 2013-2023 only. Same two helpers serve
# X (3D sequences) and y (2D targets) -- `is_sequence` picks the reshape path.

def fit_city_scalers(df, cols):
    train = df[df[YEAR_COL] <= TRAIN_END_YEAR]
    return {pid: MinMaxScaler().fit(g[cols].values) for pid, g in train.groupby(ENTITY_COL)}


def apply_scaler(arr, ent_ids, scalers, is_sequence):
    out = np.zeros_like(arr, dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        scaler = scalers[pid]
        if is_sequence:
            n, w, c = arr[mask].shape
            out[mask] = scaler.transform(arr[mask].reshape(-1, c)).reshape(n, w, c)
        else:
            out[mask] = scaler.transform(arr[mask])
    return out


def inverse_full(y_scaled, ent_ids, scalers):
    """Un-scale all 4 target columns at once (MTL predictions)."""
    out = np.zeros_like(y_scaled, dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        out[mask] = scalers[pid].inverse_transform(y_scaled[mask])
    return out


def inverse_col(y_scaled_col, ent_ids, scalers, col_index):
    """Un-scale a single target column (STL predictions). MinMaxScaler needs
    all 4 columns to invert, so pad the other 3 with zeros and discard them."""
    out = np.zeros(len(y_scaled_col), dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        scaler = scalers[pid]
        dummy = np.zeros((int(mask.sum()), scaler.n_features_in_))
        dummy[:, col_index] = y_scaled_col[mask]
        out[mask] = scaler.inverse_transform(dummy)[:, col_index]
    return out

In [11]:
# ============================================================= MODELS =====

def build_lstm_model(n_features, window_size, n_cities, target_names):
    """Shared backbone: sequence -> LSTM -> concat with city embedding ->
    Dense (this Dense layer is what lets each city get its own effective
    slope, not just an offset). One Dense(1) head per name in target_names:
    pass all 4 TARGET_VARS for the MTL model (hard parameter sharing across
    tasks), or a single target for an STL model."""
    seq_in = Input(shape=(window_size, n_features), name="sequence_input")
    ent_in = Input(shape=(1,), name="entity_input")

    lstm_out = LSTM(LSTM_UNITS, activation="tanh")(seq_in)
    emb = Flatten()(Embedding(n_cities, EMBEDDING_DIM)(ent_in))
    hidden = Dense(SHARED_DENSE_UNITS, activation="relu")(Concatenate()([lstm_out, emb]))

    heads = [Dense(1, name=f"out_{t}")(hidden) for t in target_names]
    outputs = heads if len(heads) > 1 else heads[0]  # STL: unwrap single output

    model = Model([seq_in, ent_in], outputs)
    model.compile(optimizer="adam", loss="mse")
    return model


In [12]:
# ======================================================= ARIMA BASELINE ===

def arima_forecast_all(df, target_var):
    """Per-city ARIMA(1,1,0) on that city's own 2013-2023 history, one step
    ahead. Falls back to a naive "repeat last value" forecast if the fit
    fails. `df` is assumed pre-filtered to complete-data cities, so this
    never sees a missing value; keyed by the original city id."""
    forecasts = {}
    train = df[df[YEAR_COL] <= TRAIN_END_YEAR]
    for pid, g in train.groupby(ENTITY_COL):
        series = g.sort_values(YEAR_COL)[target_var].astype(float).values
        try:
            fitted = ARIMA(series, order=(1, 1, 0)).fit()
            pred = float(np.asarray(fitted.forecast(steps=1))[0])
        except Exception:
            pred = float(series[-1])
        forecasts[pid] = pred
    return forecasts


In [13]:
# ============================================================= METRICS ====

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def smape(y_true, y_pred):
    """Symmetric MAPE (%); a 0/0 pair contributes 0 error instead of NaN."""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(y_true) + np.abs(y_pred)
    out = np.zeros_like(y_true)
    mask = denom != 0
    out[mask] = 2.0 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]
    return float(np.mean(out)) * 100


def med_ae(y_true, y_pred):
    return float(np.median(np.abs(y_true - y_pred)))


def med_smape(y_true, y_pred):
    """Symmetric MedAPE (%); a 0/0 pair contributes 0 error instead of NaN."""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(y_true) + np.abs(y_pred)
    out = np.zeros_like(y_true)
    mask = denom != 0
    out[mask] = 2.0 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]
    return float(np.median(out)) * 100


In [14]:
# =============================================================== MAIN =====

df = load_data(DATA_PATH)
n_total_cities = df[ENTITY_COL].nunique()
 
internal_cities = eligible_cities(df, TARGET_VARS)
full_cities = eligible_cities(df, TARGET_VARS + EXTERNAL_VARS)
print(f"{len(internal_cities)}/{n_total_cities} cities have complete target "
      f"data -> used by ARIMA, MTL-internal, STL-internal")
print(f"{len(full_cities)}/{n_total_cities} cities have complete target+external "
      f"data -> used by MTL-full, STL-full")
 
monitor = "val_loss" if (USE_VALIDATION and VAL_YEARS > 0) else "loss"
early_stop = EarlyStopping(monitor=monitor, patience=PATIENCE, restore_best_weights=True)
n_targets = len(TARGET_VARS)
results = []
 
# ---- ARIMA baseline (complete-target cities only) ----
df_arima = df[df[ENTITY_COL].isin(internal_cities)]
print(f"\nFitting ARIMA baselines on {len(internal_cities)} cities...")
arima_by_target = {t: arima_forecast_all(df_arima, t) for t in TARGET_VARS}
test_rows = df_arima[df_arima[YEAR_COL] == FORECAST_YEAR].set_index(ENTITY_COL)
 
for target in TARGET_VARS:
    y_true = test_rows[target].values
    y_pred = np.array([arima_by_target[target].get(pid, np.nan) for pid in test_rows.index])
    valid = ~np.isnan(y_pred)
    results.append({"target_variable": target, "model": "ARIMA",
                     "n_cities_used": len(internal_cities),
                     "n_test_cities": int(valid.sum()),
                     "MedAE": round(med_ae(y_true[valid], y_pred[valid]), 2),
                     "sMdAPE_%": round(med_smape(y_true[valid], y_pred[valid]), 2)})
 
# ---- LSTM runs: {full, internal} feature sets x {MTL, STL} ----
for feature_label, feature_cols in FEATURE_SETS.items():
    cities = full_cities if feature_label == "full" else internal_cities
    n_cities = len(cities)
    print(f"\nTraining LSTM models on '{feature_label}' features ({n_cities} cities)...")
 
    df_sub = df[df[ENTITY_COL].isin(cities)].copy()
    pid_to_idx = {pid: i for i, pid in enumerate(cities)}
    df_sub[ENTITY_COL] = df_sub[ENTITY_COL].map(pid_to_idx)
 
    X_seq, X_ent, y, year = build_sequences(df_sub, feature_cols, WINDOW_SIZE)
    train_mask, test_mask = year <= TRAIN_END_YEAR, year == FORECAST_YEAR
 
    x_scalers = fit_city_scalers(df_sub, feature_cols)
    y_scalers = fit_city_scalers(df_sub, TARGET_VARS)
    X_tr = apply_scaler(X_seq[train_mask], X_ent[train_mask], x_scalers, is_sequence=True)
    X_te = apply_scaler(X_seq[test_mask], X_ent[test_mask], x_scalers, is_sequence=True)
    y_tr = apply_scaler(y[train_mask], X_ent[train_mask], y_scalers, is_sequence=False)
    ent_tr, ent_te, y_te = X_ent[train_mask], X_ent[test_mask], y[test_mask]
    year_tr = year[train_mask]
    n_features = len(feature_cols)
 
    use_val = USE_VALIDATION and VAL_YEARS > 0
    if use_val:
        fit_sel = year_tr <= (TRAIN_END_YEAR - VAL_YEARS)
        val_sel = ~fit_sel
    else:
        fit_sel = np.ones_like(year_tr, dtype=bool)
        val_sel = np.zeros_like(year_tr, dtype=bool)
 
    X_fit, ent_fit, y_fit = X_tr[fit_sel], ent_tr[fit_sel], y_tr[fit_sel]
    X_val, ent_val, y_val = X_tr[val_sel], ent_tr[val_sel], y_tr[val_sel]
    if use_val and len(y_val) == 0:
        print(f"  [warning] VAL_YEARS={VAL_YEARS} left no validation rows for "
              f"'{feature_label}' -- falling back to training-loss early stopping.")
        use_val = False
 
    # MTL: one model, four heads, trained jointly
    mtl = build_lstm_model(n_features, WINDOW_SIZE, n_cities, TARGET_VARS)
    mtl_fit_kwargs = dict(epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=[early_stop])
    if use_val:
        mtl_fit_kwargs["validation_data"] = ([X_val, ent_val], [y_val[:, j] for j in range(n_targets)])
    mtl.fit([X_fit, ent_fit], [y_fit[:, j] for j in range(n_targets)], **mtl_fit_kwargs)
    mtl_pred = inverse_full(np.column_stack(mtl.predict([X_te, ent_te], verbose=0)), ent_te, y_scalers)
 
    # STL: four independent single-head models
    stl_pred = np.zeros_like(mtl_pred)
    for j, target in enumerate(TARGET_VARS):
        stl = build_lstm_model(n_features, WINDOW_SIZE, n_cities, [target])
        stl_fit_kwargs = dict(epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=[early_stop])
        if use_val:
            stl_fit_kwargs["validation_data"] = ([X_val, ent_val], y_val[:, j])
        stl.fit([X_fit, ent_fit], y_fit[:, j], **stl_fit_kwargs)
        pred_s = stl.predict([X_te, ent_te], verbose=0).flatten()
        stl_pred[:, j] = inverse_col(pred_s, ent_te, y_scalers, j)
 
    for j, target in enumerate(TARGET_VARS):
        y_true = y_te[:, j]
        for model_name, y_pred in [(f"MTL-LSTM ({feature_label})", mtl_pred[:, j]),
                                    (f"STL-LSTM ({feature_label})", stl_pred[:, j])]:
            valid = ~np.isnan(y_pred)
            results.append({"target_variable": target, "model": model_name,
                             "n_cities_used": n_cities,
                             "n_test_cities": int(valid.sum()),
                             "MedAE": round(med_ae(y_true[valid], y_pred[valid]), 2),
                             "sMdAPE_%": round(med_smape(y_true[valid], y_pred[valid]), 2)})
 
results_df = pd.DataFrame(results)
print(f"\nDone. {len(results_df)} rows stored in `results_df`. Run the report cell next.")
 

258/274 cities have complete target data -> used by ARIMA, MTL-internal, STL-internal
257/274 cities have complete target+external data -> used by MTL-full, STL-full

Fitting ARIMA baselines on 258 cities...

Training LSTM models on 'full' features (257 cities)...

Training LSTM models on 'internal' features (258 cities)...

Done. 30 rows stored in `results_df`. Run the report cell next.


In [15]:
# ===========================================================  REPORT ===
pd.set_option("display.width", 120)
 
MODEL_ORDER_NOTE = ("Note: ARIMA / *-internal models and *-full models are NOT "
                     "scored on the same set of cities -- see n_cities_used row.")
 
# Fixed so every target's table uses the same column order -- makes the
# tables directly comparable side by side instead of re-sorting per target.
MODEL_ORDER = ["ARIMA", "MTL-LSTM (internal)", "STL-LSTM (internal)",
               "MTL-LSTM (full)", "STL-LSTM (full)"]
 
HYPOTHESES = [
    ("H1", "MTL-LSTM (full) < ARIMA",               "MTL-LSTM (full)", "ARIMA"),
    ("H2", "MTL-LSTM (full) < STL-LSTM (full)",      "MTL-LSTM (full)", "STL-LSTM (full)"),
    ("H3", "MTL-LSTM (full) < MTL-LSTM (internal)",  "MTL-LSTM (full)", "MTL-LSTM (internal)"),
    ("H4", "STL-LSTM (full) < STL-LSTM (internal)",  "STL-LSTM (full)", "STL-LSTM (internal)"),
]
 
overview_rows = []
 
for target in TARGET_VARS:
    sub = results_df[results_df["target_variable"] == target].set_index("model")
    sub = sub[["n_cities_used", "n_test_cities", "MedAE", "sMdAPE_%"]]
    sub = sub.reindex([m for m in MODEL_ORDER if m in sub.index])  # unified column order
 
    best_model = sub["MedAE"].idxmin()
    print(f"\n=== {target} (best: {best_model}) ===")
    print(sub.T.to_string())
 
    overview_row = {"target_variable": target}
    for h_id, label, model_a, model_b in HYPOTHESES:
        if model_a in sub.index and model_b in sub.index:
            supported = bool(sub.loc[model_a, "MedAE"] < sub.loc[model_b, "MedAE"])
            print(f"  {h_id} ({label}): "
                  f"{'SUPPORTED' if supported else 'NOT SUPPORTED'} "
                  f"[{sub.loc[model_a, 'MedAE']:.2f} vs {sub.loc[model_b, 'MedAE']:.2f}]")
            overview_row[h_id] = "Y" if supported else "N"
        else:
            print(f"  {h_id} ({label}): N/A (missing model rows)")
            overview_row[h_id] = "N/A"
    overview_rows.append(overview_row)
 
overview_df = pd.DataFrame(overview_rows).set_index("target_variable")
print("\n=== Hypothesis overview (Y = supported, N = not supported, per target) ===")
print(overview_df.to_string())
print(f"\n{MODEL_ORDER_NOTE}")


=== opr_ratio_gn (best: MTL-LSTM (internal)) ===
model           ARIMA  MTL-LSTM (internal)  STL-LSTM (internal)  MTL-LSTM (full)  STL-LSTM (full)
n_cities_used  258.00               258.00               258.00           257.00           257.00
n_test_cities  258.00               258.00               258.00           257.00           257.00
MedAE            0.13                 0.12                 0.12             0.15             0.15
sMdAPE_%        11.67                10.33                10.69            13.70            13.07
  H1 (MTL-LSTM (full) < ARIMA): NOT SUPPORTED [0.15 vs 0.13]
  H2 (MTL-LSTM (full) < STL-LSTM (full)): NOT SUPPORTED [0.15 vs 0.15]
  H3 (MTL-LSTM (full) < MTL-LSTM (internal)): NOT SUPPORTED [0.15 vs 0.12]
  H4 (STL-LSTM (full) < STL-LSTM (internal)): NOT SUPPORTED [0.15 vs 0.12]

=== opr_ratio_ep (best: ARIMA) ===
model          ARIMA  MTL-LSTM (internal)  STL-LSTM (internal)  MTL-LSTM (full)  STL-LSTM (full)
n_cities_used  258.0               258.00    

In [14]:
"""
Direct (non-recursive) 3-year-ahead forecasting: ARIMA vs MTL/STL
entity-embedded LSTMs, each over {full, internal} feature sets.

Train: 2013-2021. Test: 2022, 2023, 2024 (direct horizons h=1,2,3).

Direct = one fixed input window (2019-2021, all actual) feeds dedicated
output heads per horizon step. No prediction is ever fed back in as an
input for another prediction (that's the recursive alternative).

Training examples for the LSTM heads use window-end years Y with
Y + HORIZON <= TRAIN_END_YEAR (Y <= 2018 here), so no 2022+ data is used
anywhere in training -- only ~4 examples/city as a result; expect a noisy
fit for a 9-year history. Widen MAX_ORIGIN_YEAR to trade that isolation
for more training volume if needed.
"""

np.random.seed(42)
tf.random.set_seed(42)

DATA_PATH = "variable.xlsx"
TARGET_VARS = ["opr_ratio_gn", "opr_ratio_ep", "cash_ratio_totasst","totdebt_to_asst","capital_to_asst", "funded_ratio_total"]
EXTERNAL_VARS = ["ln_pop_city", "ln_curgdp", "ln_psnl_incm", "employment", "ln_med_homevalue",
                    "property_rel","intg_rev_rel",
                    "disaster_event", "flood_dmg_tot",
                    "under_18%", "age_65_plus%", "white%", "bachelors%", "disability%", "poverty%"
                    ]
ENTITY_COL, YEAR_COL = "pid", "year"

WINDOW_SIZE = 3
TRAIN_END_YEAR = 2021
FORECAST_YEARS = [2022, 2023, 2024]
HORIZON = len(FORECAST_YEARS)
MAX_ORIGIN_YEAR = TRAIN_END_YEAR - HORIZON   # last usable training window-end year

USE_VALIDATION, VAL_YEARS = True, 1
EMBEDDING_DIM, LSTM_UNITS, SHARED_DENSE_UNITS = 4, 16, 16
EPOCHS, BATCH_SIZE, PATIENCE = 200, 32, 15

FEATURE_SETS = {"full": TARGET_VARS + EXTERNAL_VARS, "internal": TARGET_VARS}

# =========================================================== DATA PREP ====

def load_data(path):
    df = pd.read_excel(path)
    keep = [ENTITY_COL, "city", YEAR_COL] + TARGET_VARS + EXTERNAL_VARS
    return df[keep].sort_values([ENTITY_COL, YEAR_COL]).reset_index(drop=True)


def eligible_cities(df, cols):
    incomplete = df.loc[df[cols].isnull().any(axis=1), ENTITY_COL].unique()
    return sorted(set(df[ENTITY_COL].unique()) - set(incomplete))


def build_direct_sequences(df, feature_cols, window_size, horizon, max_origin_year):
    """One example per (city, window-end year Y <= max_origin_year): X =
    actual window ending at Y, y = actual targets at Y+1..Y+horizon."""
    X_seq, X_ent, Y_multi, origin_year = [], [], [], []
    for pid, g in df.groupby(ENTITY_COL):
        g = g.sort_values(YEAR_COL)
        feats, targs, years = g[feature_cols].values, g[TARGET_VARS].values, g[YEAR_COL].values
        for i in range(len(g) - window_size - horizon + 1):
            end_idx = i + window_size - 1
            Y = years[end_idx]
            if Y > max_origin_year:
                continue
            span = list(years[i:i + window_size + horizon])
            if span != list(range(years[i], years[i] + window_size + horizon)):
                continue  # non-contiguous years -> skip
            X_seq.append(feats[i:i + window_size])
            X_ent.append(pid)
            Y_multi.append(targs[end_idx + 1:end_idx + 1 + horizon])
            origin_year.append(Y)
    return (np.array(X_seq, dtype="float32"), np.array(X_ent, dtype="int32"),
            np.array(Y_multi, dtype="float32"), np.array(origin_year))


def build_test_window(df, feature_cols, window_size, train_end_year, pids):
    """Single actual window per city: [train_end_year-window_size+1, train_end_year]."""
    wanted = list(range(train_end_year - window_size + 1, train_end_year + 1))
    X_seq, ent_ids = [], []
    for pid in pids:
        g = df[df[ENTITY_COL] == pid].sort_values(YEAR_COL)
        g = g[g[YEAR_COL].isin(wanted)]
        if list(g[YEAR_COL].values) != wanted:
            continue
        X_seq.append(g[feature_cols].values)
        ent_ids.append(pid)
    X_seq = np.array(X_seq, dtype="float32") if X_seq else np.zeros((0, window_size, len(feature_cols)), "float32")
    return X_seq, np.array(ent_ids, dtype="int32")

# ======================================================= CITY SCALING =====

def fit_city_scalers(df, cols):
    train = df[df[YEAR_COL] <= TRAIN_END_YEAR]
    return {pid: MinMaxScaler().fit(g[cols].values) for pid, g in train.groupby(ENTITY_COL)}


def apply_scaler(arr, ent_ids, scalers, is_sequence):
    out = np.zeros_like(arr, dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        if is_sequence:
            n, w, c = arr[mask].shape
            out[mask] = scalers[pid].transform(arr[mask].reshape(-1, c)).reshape(n, w, c)
        else:
            out[mask] = scalers[pid].transform(arr[mask])
    return out


def inverse_full(y_scaled, ent_ids, scalers):
    out = np.zeros_like(y_scaled, dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        out[mask] = scalers[pid].inverse_transform(y_scaled[mask])
    return out


def inverse_col(y_scaled_col, ent_ids, scalers, col_index):
    out = np.zeros(len(y_scaled_col), dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        scaler = scalers[pid]
        dummy = np.zeros((int(mask.sum()), scaler.n_features_in_))
        dummy[:, col_index] = y_scaled_col[mask]
        out[mask] = scaler.inverse_transform(dummy)[:, col_index]
    return out

# ============================================================= MODELS =====

def build_direct_lstm_model(n_features, window_size, n_cities, head_names):
    """Shared LSTM+embedding backbone; one Dense(1) head per name in
    head_names, all reading the same window (no head feeds another)."""
    seq_in = Input(shape=(window_size, n_features), name="sequence_input")
    ent_in = Input(shape=(1,), name="entity_input")
    lstm_out = LSTM(LSTM_UNITS, activation="tanh")(seq_in)
    emb = Flatten()(Embedding(n_cities, EMBEDDING_DIM)(ent_in))
    hidden = Dense(SHARED_DENSE_UNITS, activation="relu")(Concatenate()([lstm_out, emb]))
    heads = [Dense(1, name=f"out_{name}")(hidden) for name in head_names]
    model = Model([seq_in, ent_in], heads if len(heads) > 1 else heads[0])
    model.compile(optimizer="adam", loss="mse")
    return model

# ======================================================= ARIMA BASELINE ===

def arima_forecast_multi(df, target_var, horizon):
    """Fit once on 2013-TRAIN_END_YEAR; one forecast(steps=horizon) call
    returns all steps directly -- already non-recursive w.r.t. this script."""
    forecasts = {}
    train = df[df[YEAR_COL] <= TRAIN_END_YEAR]
    for pid, g in train.groupby(ENTITY_COL):
        series = g.sort_values(YEAR_COL)[target_var].astype(float).values
        try:
            pred = np.asarray(ARIMA(series, order=(1, 1, 0)).fit().forecast(steps=horizon), dtype=float)
        except Exception:
            pred = np.repeat(series[-1], horizon)
        forecasts[pid] = pred
    return forecasts

# ============================================================= METRICS ====
# Standard, mean-based definitions. Used as-is for the per-year diagnostic
# rows (pooled across cities within a single forecast year). Multistep
# (across-horizon) averaging and cross-city median aggregation are handled
# separately below in score() / summarize_median_across_cities() -- neither
# of those change these formulas.

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def smape(y_true, y_pred):
    """Symmetric MAPE (%); a 0/0 pair contributes 0 error instead of NaN."""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(y_true) + np.abs(y_pred)
    out = np.zeros_like(y_true)
    mask = denom != 0
    out[mask] = 2.0 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]
    return float(np.mean(out)) * 100


def per_city_abs_error(y_true, y_pred):
    return np.abs(np.asarray(y_true, float) - np.asarray(y_pred, float))


def per_city_smape_term(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(y_true) + np.abs(y_pred)
    out = np.zeros_like(y_true)
    mask = denom != 0
    out[mask] = 2.0 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]
    return out * 100

# ================================================ SCORING / AGGREGATION ===
# score() appends both the mean-based diagnostic rows (per year + "ALL")
# and each city's own mean-across-horizons error. summarize_median_across_
# cities() is the only place np.median is used -- collapsing the city
# dimension for the final cross-model comparison.

def score(df_sub, target, preds, n_cities, results, per_city_records, model_name):
    """
    preds: {forecast_year: {pid: prediction}}

    - Appends per-year diagnostic rows (mean over cities, standard MAE/
      sMAPE) plus an "ALL" row (mean over years of those -- still a mean,
      unchanged from before).
    - Separately fills per_city_records with each city's own error at
      each horizon, so the caller can average across horizons (mean) per
      city and then take the median across cities for final comparison.
    """
    year_maes, year_smapes = [], []
    # pid -> list of (abs_error, smape_term), one entry per available horizon
    city_errors = {}

    for h, fy in enumerate(FORECAST_YEARS):
        true_rows = df_sub[df_sub[YEAR_COL] == fy].set_index(ENTITY_COL)
        pid_pred = preds.get(fy, {})
        pids = [p for p in true_rows.index if p in pid_pred]
        y_true = true_rows.loc[pids, target].values
        y_pred = np.array([pid_pred[p] for p in pids])

        m, s = mae(y_true, y_pred), smape(y_true, y_pred)
        year_maes.append(m); year_smapes.append(s)
        results.append({"target_variable": target, "model": model_name, "forecast_year": fy,
                         "n_cities_used": n_cities, "n_test_cities": len(pids),
                         "MAE": round(m, 2), "sMAPE_%": round(s, 2)})

        abs_err = per_city_abs_error(y_true, y_pred)
        sm_err = per_city_smape_term(y_true, y_pred)
        for pid, ae, se in zip(pids, abs_err, sm_err):
            city_errors.setdefault(pid, []).append((ae, se))

    results.append({"target_variable": target, "model": model_name, "forecast_year": "ALL",
                     "n_cities_used": n_cities, "n_test_cities": np.nan,
                     "MAE": round(float(np.mean(year_maes)), 2),
                     "sMAPE_%": round(float(np.mean(year_smapes)), 2)})

    # Keep only cities with an error at every horizon, so each city's
    # multistep average is over the same number of steps as every other
    # city's -- then MEAN across that city's own horizons.
    for pid, errs in city_errors.items():
        if len(errs) != HORIZON:
            continue
        ae_arr = np.array([e[0] for e in errs])
        se_arr = np.array([e[1] for e in errs])
        per_city_records.append({
            "target_variable": target, "model": model_name, "pid": pid,
            "city_MAE": float(np.mean(ae_arr)),        # mean across horizons
            "city_sMAPE_%": float(np.mean(se_arr)),    # mean across horizons
        })


def summarize_median_across_cities(per_city_records):
    """The only place np.median is used: collapsing the city dimension of
    each city's own (already horizon-averaged) error, to get one robust
    number per (target, model) for the final cross-model comparison."""
    pc_df = pd.DataFrame(per_city_records)
    summary = pc_df.groupby(["target_variable", "model"]).agg(
        MedAE=("city_MAE", "median"),
        MedsMAPE_pct=("city_sMAPE_%", "median"),
        n_cities=("pid", "nunique"),
    ).rename(columns={"MedsMAPE_pct": "MedsMAPE_%"}).round(2).reset_index()
    return summary


# %% ======================================================= CELL 1: RUN ===
# Trains ARIMA + the four LSTM variants for the 3-year direct-horizon setup
# and stores raw results in memory only:
#   - results_df       : per (target, model, forecast_year) + "ALL" row,
#                         standard mean-based MAE/sMAPE (diagnostic, pooled
#                         across cities, mean across horizons)
#   - per_city_records  : each city's own error at each horizon (mean
#                         across horizons per city) -- feeds summary_df
#   - summary_df        : per (target, model), MEDIAN across cities of each
#                         city's mean-over-horizons error -- this is what
#                         Cell 2 uses for the actual model comparison
# No printing of tables, no summary math -- that's Cell 2's job.
# Assumes load_data, eligible_cities, build_direct_sequences,
# build_test_window, fit_city_scalers, apply_scaler, inverse_full,
# inverse_col, build_direct_lstm_model, arima_forecast_multi are already
# defined (DATA PREP / CITY SCALING / MODELS / ARIMA BASELINE sections above).

df = load_data(DATA_PATH)
n_total = df[ENTITY_COL].nunique()
internal_cities = eligible_cities(df, TARGET_VARS)
full_cities = eligible_cities(df, TARGET_VARS + EXTERNAL_VARS)
print(f"{len(internal_cities)}/{n_total} cities eligible (targets) -> ARIMA, *-internal")
print(f"{len(full_cities)}/{n_total} cities eligible (targets+externals) -> *-full")

early_stop = EarlyStopping(monitor="val_loss" if (USE_VALIDATION and VAL_YEARS > 0) else "loss",
                            patience=PATIENCE, restore_best_weights=True)
results = []
per_city_records = []

# ---- ARIMA ----
df_arima = df[df[ENTITY_COL].isin(internal_cities)]
arima_by_target = {t: arima_forecast_multi(df_arima, t, HORIZON) for t in TARGET_VARS}
for target in TARGET_VARS:
    preds = {fy: {pid: arima_by_target[target][pid][h] for pid in arima_by_target[target]}
             for h, fy in enumerate(FORECAST_YEARS)}
    score(df_arima, target, preds, len(internal_cities), results, per_city_records, "ARIMA")

# ---- LSTM: {full, internal} x {MTL, STL} ----
for feature_label, feature_cols in FEATURE_SETS.items():
    cities = full_cities if feature_label == "full" else internal_cities
    n_cities, n_features = len(cities), len(feature_cols)
    print(f"\n'{feature_label}': {n_cities} cities, {n_features} features")

    df_sub = df[df[ENTITY_COL].isin(cities)].copy()
    pid_map = {pid: i for i, pid in enumerate(cities)}
    df_sub[ENTITY_COL] = df_sub[ENTITY_COL].map(pid_map)
    recoded_pids = list(range(n_cities))

    X_seq, X_ent, Y_multi, origin_year = build_direct_sequences(
        df_sub, feature_cols, WINDOW_SIZE, HORIZON, MAX_ORIGIN_YEAR)
    print(f"  {len(origin_year)} direct training examples "
          f"({len(origin_year)/max(n_cities,1):.1f}/city, origins {sorted(set(origin_year.tolist()))})")

    x_scalers = fit_city_scalers(df_sub, feature_cols)
    y_scalers = fit_city_scalers(df_sub, TARGET_VARS)
    X_tr = apply_scaler(X_seq, X_ent, x_scalers, is_sequence=True)
    Y_tr = np.zeros_like(Y_multi, dtype=float)
    for h in range(HORIZON):
        Y_tr[:, h, :] = apply_scaler(Y_multi[:, h, :], X_ent, y_scalers, is_sequence=False)

    use_val = USE_VALIDATION and VAL_YEARS > 0
    fit_sel = origin_year <= (MAX_ORIGIN_YEAR - VAL_YEARS) if use_val else np.ones_like(origin_year, bool)
    val_sel = ~fit_sel if use_val else np.zeros_like(origin_year, bool)
    X_fit, ent_fit, Y_fit = X_tr[fit_sel], X_ent[fit_sel], Y_tr[fit_sel]
    X_val, ent_val, Y_val = X_tr[val_sel], X_ent[val_sel], Y_tr[val_sel]
    if use_val and len(Y_val) == 0:
        use_val = False

    X_test, ent_test = build_test_window(df_sub, feature_cols, WINDOW_SIZE, TRAIN_END_YEAR, recoded_pids)
    X_test_scaled = apply_scaler(X_test, ent_test, x_scalers, is_sequence=True) if len(ent_test) else X_test
    fit_kwargs = dict(epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=[early_stop])

    # MTL: 12 heads (target-major, horizon-minor)
    mtl_heads = [f"{t}_h{h+1}" for t in TARGET_VARS for h in range(HORIZON)]
    mtl = build_direct_lstm_model(n_features, WINDOW_SIZE, n_cities, mtl_heads)
    y_fit_list = [Y_fit[:, h, j] for j in range(len(TARGET_VARS)) for h in range(HORIZON)]
    kw = dict(fit_kwargs)
    if use_val:
        kw["validation_data"] = ([X_val, ent_val], [Y_val[:, h, j] for j in range(len(TARGET_VARS)) for h in range(HORIZON)])
    mtl.fit([X_fit, ent_fit], y_fit_list, **kw)

    mtl_preds = {t: {} for t in TARGET_VARS}
    if len(ent_test):
        raw = mtl.predict([X_test_scaled, ent_test], verbose=0)
        raw = raw if isinstance(raw, list) else [raw]
        for h, fy in enumerate(FORECAST_YEARS):
            mat = np.column_stack([raw[j * HORIZON + h].flatten() for j in range(len(TARGET_VARS))])
            mat = inverse_full(mat, ent_test, y_scalers)
            for j, target in enumerate(TARGET_VARS):
                mtl_preds[target][fy] = dict(zip(ent_test.tolist(), mat[:, j].tolist()))

    # STL: 4 networks x 3 heads
    stl_preds = {t: {} for t in TARGET_VARS}
    for j, target in enumerate(TARGET_VARS):
        stl = build_direct_lstm_model(n_features, WINDOW_SIZE, n_cities, [f"{target}_h{h+1}" for h in range(HORIZON)])
        kw = dict(fit_kwargs)
        if use_val:
            kw["validation_data"] = ([X_val, ent_val], [Y_val[:, h, j] for h in range(HORIZON)])
        stl.fit([X_fit, ent_fit], [Y_fit[:, h, j] for h in range(HORIZON)], **kw)
        if len(ent_test):
            raw = stl.predict([X_test_scaled, ent_test], verbose=0)
            raw = raw if isinstance(raw, list) else [raw]
            for h, fy in enumerate(FORECAST_YEARS):
                pred = inverse_col(raw[h].flatten(), ent_test, y_scalers, j)
                stl_preds[target][fy] = dict(zip(ent_test.tolist(), pred.tolist()))

    for target in TARGET_VARS:
        score(df_sub, target, mtl_preds[target], n_cities, results, per_city_records, f"MTL-LSTM ({feature_label})")
        score(df_sub, target, stl_preds[target], n_cities, results, per_city_records, f"STL-LSTM ({feature_label})")

results_df = pd.DataFrame(results)
summary_df = summarize_median_across_cities(per_city_records)
print(f"\nDone. {len(results_df)} diagnostic rows in `results_df`, "
      f"{len(summary_df)} rows in `summary_df`. Run the report cell next.")


# %% ==================================================== CELL 2: REPORT ===
# Uses `results_df` and `summary_df` directly from memory (run Cell 1 first,
# in the same kernel session). No cross-target averaging: each target
# variable gets its own comparison table -- MEDIAN across cities of each
# city's mean-over-horizons error, the metric summarize_median_across_cities
# already computed -- and its own H1-H4 hypothesis check.

pd.set_option("display.width", 120)

MODEL_ORDER_NOTE = ("Note: ARIMA / *-internal models and *-full models are NOT "
                     "scored on the same set of cities -- see n_cities row.")

# Fixed so every target's table uses the same column order -- makes the
# tables directly comparable side by side instead of re-sorting per target.
MODEL_ORDER = ["ARIMA", "MTL-LSTM (internal)", "STL-LSTM (internal)",
               "MTL-LSTM (full)", "STL-LSTM (full)"]

HYPOTHESES = [
    ("H1", "MTL-LSTM (full) < ARIMA",               "MTL-LSTM (full)", "ARIMA"),
    ("H2", "MTL-LSTM (full) < STL-LSTM (full)",      "MTL-LSTM (full)", "STL-LSTM (full)"),
    ("H3", "MTL-LSTM (full) < MTL-LSTM (internal)",  "MTL-LSTM (full)", "MTL-LSTM (internal)"),
    ("H4", "STL-LSTM (full) < STL-LSTM (internal)",  "STL-LSTM (full)", "STL-LSTM (internal)"),
]

overview_rows = []

for target in TARGET_VARS:
    sub = summary_df[summary_df["target_variable"] == target].set_index("model")
    sub = sub[["n_cities", "MedAE", "MedsMAPE_%"]]
    sub = sub.reindex([m for m in MODEL_ORDER if m in sub.index])  # unified column order

    best_model = sub["MedAE"].idxmin()
    print(f"\n=== {target} (best: {best_model}; median across cities of each city's "
          f"mean-over-{HORIZON}-horizons error) ===")
    print(sub.T.to_string())

    overview_row = {"target_variable": target}
    for h_id, label, model_a, model_b in HYPOTHESES:
        if model_a in sub.index and model_b in sub.index:
            supported = bool(sub.loc[model_a, "MedAE"] < sub.loc[model_b, "MedAE"])
            print(f"  {h_id} ({label}): "
                  f"{'SUPPORTED' if supported else 'NOT SUPPORTED'} "
                  f"[{sub.loc[model_a, 'MedAE']:.2f} vs {sub.loc[model_b, 'MedAE']:.2f}]")
            overview_row[h_id] = "Y" if supported else "N"
        else:
            print(f"  {h_id} ({label}): N/A (missing model rows)")
            overview_row[h_id] = "N/A"
    overview_rows.append(overview_row)

overview_df = pd.DataFrame(overview_rows).set_index("target_variable")
print("\n=== Hypothesis overview (Y = supported, N = not supported, per target) ===")
print(overview_df.to_string())
print(f"\n{MODEL_ORDER_NOTE}")

# ---- Optional: per-year diagnostic detail (mean-based, pooled across ----
# ---- cities) if you want to see how error grows with horizon h=1,2,3 ----
SHOW_PER_YEAR_DIAGNOSTICS = False
if SHOW_PER_YEAR_DIAGNOSTICS:
    for target in TARGET_VARS:
        print(f"\n--- {target}: per-year diagnostic (mean over cities, standard MAE/sMAPE) ---")
        diag = results_df[results_df["target_variable"] == target]
        diag = diag.pivot(index="model", columns="forecast_year",
                           values=["MAE", "sMAPE_%"])
        print(diag.to_string())

258/274 cities eligible (targets) -> ARIMA, *-internal
257/274 cities eligible (targets+externals) -> *-full

'full': 257 cities, 21 features
  1028 direct training examples (4.0/city, origins [2015, 2016, 2017, 2018])

'internal': 258 cities, 6 features
  1032 direct training examples (4.0/city, origins [2015, 2016, 2017, 2018])

Done. 120 diagnostic rows in `results_df`, 30 rows in `summary_df`. Run the report cell next.

=== opr_ratio_gn (best: ARIMA; median across cities of each city's mean-over-3-horizons error) ===
model        ARIMA  MTL-LSTM (internal)  STL-LSTM (internal)  MTL-LSTM (full)  STL-LSTM (full)
n_cities    258.00               258.00               258.00           257.00           257.00
MedAE         0.14                 0.19                 0.17             0.19             0.16
MedsMAPE_%   12.36                16.78                15.73            18.22            13.93
  H1 (MTL-LSTM (full) < ARIMA): NOT SUPPORTED [0.19 vs 0.14]
  H2 (MTL-LSTM (full) < STL-LSTM

From now on, we're going to use tree model; 

In [16]:

# ============================================================== CONFIG ====
DATA_PATH = "variable.xlsx"
TARGET_VARS = ["opr_ratio_gn", "opr_ratio_ep", "cash_ratio_totasst","totdebt_to_asst","capital_to_asst", "funded_ratio_total"]
EXTERNAL_VARS = ["ln_pop_city", "ln_curgdp", "ln_psnl_incm", "employment", "ln_med_homevalue",
                    "property_rel", "intg_rev_rel",
                    "disaster_event", "flood_dmg_tot",
                    "under_18%", "age_65_plus%", "white%", "bachelors%", "disability%", "poverty%"
                    ]
ENTITY_COL, YEAR_COL = "pid", "year"

WINDOW_SIZE = 3           # years of history each model sees before predicting
TRAIN_END_YEAR = 2023     # last year usable as a TRAINING target
FORECAST_YEAR = 2024      # the held-out year to score against
# Same one-step-ahead caveat as the LSTM script: this is not a multi-step
# forecaster. Whatever FORECAST_YEAR is set to, the model's input window is
# the WINDOW_SIZE actual years immediately before it.

# Validation split: used for (a) XGBoost early stopping, and (b) choosing k
# for both KNN variants. The most recent VAL_YEARS training years are held
# out from fitting and used only to pick when-to-stop / which-k.
USE_VALIDATION = True
VAL_YEARS = 1

# --- entity-embedding pretraining net (see docstring point 1) ---
EMBEDDING_DIM, LSTM_UNITS, SHARED_DENSE_UNITS = 4, 16, 16
EMB_EPOCHS, EMB_BATCH_SIZE, EMB_PATIENCE = 200, 32, 15

# --- XGBoost ---
XGB_N_ESTIMATORS = 500
XGB_EARLY_STOPPING_ROUNDS = 20
XGB_PARAMS = dict(tree_method="hist", max_depth=3, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, eval_metric="rmse",
                   random_state=42)

# --- KNN ---
KNN_K_GRID = [3, 5, 7, 10, 15, 20, 30]   # candidate neighbor counts to try
KNN_WEIGHTS = "distance"                  # closer neighbors count more

FEATURE_SETS = {
    "full": TARGET_VARS + EXTERNAL_VARS,
    "internal": TARGET_VARS,
}

# =========================================================== DATA PREP ====
# (unchanged from the LSTM script)

def load_data(path):
    """Read Excel and keep only the needed columns. No filling/imputation
    happens here or anywhere else in this script."""
    df = pd.read_excel(path)
    keep = [ENTITY_COL, "city", YEAR_COL] + TARGET_VARS + EXTERNAL_VARS
    return df[keep].sort_values([ENTITY_COL, YEAR_COL]).reset_index(drop=True)


def eligible_cities(df, cols):
    """Cities with NO missing value in any of `cols`, across every year
    they appear. Returns a sorted list of city ids (original, un-recoded)."""
    incomplete = df.loc[df[cols].isnull().any(axis=1), ENTITY_COL].unique()
    return sorted(set(df[ENTITY_COL].unique()) - set(incomplete))


def build_sequences(df, feature_cols, window_size):
    """Slide a window of `window_size` years per city.
    Returns X_seq (n, window, n_features), X_ent (n,), y (n, 4 targets),
    target_year (n,)."""
    X_seq, X_ent, y, target_year = [], [], [], []
    for pid, g in df.groupby(ENTITY_COL):
        g = g.sort_values(YEAR_COL)
        feats = g[feature_cols].values
        targs = g[TARGET_VARS].values
        years = g[YEAR_COL].values
        for i in range(len(g) - window_size):
            X_seq.append(feats[i:i + window_size])
            X_ent.append(pid)
            y.append(targs[i + window_size])
            target_year.append(years[i + window_size])
    return (np.array(X_seq, dtype="float32"), np.array(X_ent, dtype="int32"),
            np.array(y, dtype="float32"), np.array(target_year))


def flatten_sequences(X_seq):
    """(n, window, n_features) -> (n, window*n_features). XGBoost and KNN
    take flat feature vectors, not sequences, so the window is unrolled
    into columns (year0_var0, year0_var1, ..., year(w-1)_var(k-1))."""
    n = X_seq.shape[0]
    return X_seq.reshape(n, -1)

# ======================================================= CITY SCALING =====
# Unchanged from the LSTM script: one MinMaxScaler per city per role
# (features vs. targets), fit on 2013-2023 only, inverted with that same
# city's parameters before scoring. See the LSTM script's header comment
# for the full rationale.

def fit_city_scalers(df, cols):
    train = df[df[YEAR_COL] <= TRAIN_END_YEAR]
    return {pid: MinMaxScaler().fit(g[cols].values) for pid, g in train.groupby(ENTITY_COL)}


def apply_scaler(arr, ent_ids, scalers, is_sequence):
    out = np.zeros_like(arr, dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        scaler = scalers[pid]
        if is_sequence:
            n, w, c = arr[mask].shape
            out[mask] = scaler.transform(arr[mask].reshape(-1, c)).reshape(n, w, c)
        else:
            out[mask] = scaler.transform(arr[mask])
    return out


def inverse_full(y_scaled, ent_ids, scalers):
    out = np.zeros_like(y_scaled, dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        out[mask] = scalers[pid].inverse_transform(y_scaled[mask])
    return out


def inverse_col(y_scaled_col, ent_ids, scalers, col_index):
    out = np.zeros(len(y_scaled_col), dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        scaler = scalers[pid]
        dummy = np.zeros((int(mask.sum()), scaler.n_features_in_))
        dummy[:, col_index] = y_scaled_col[mask]
        out[mask] = scaler.inverse_transform(dummy)[:, col_index]
    return out

# ================================================ ENTITY EMBEDDING NET =====
# Trained purely to obtain a per-city vector for XGBoost/KNN to consume
# (see docstring point 1). Same architecture family as the LSTM script's
# MTL model. Nothing downstream of the Embedding layer is kept.

def build_embedding_net(n_features, window_size, n_cities):
    seq_in = Input(shape=(window_size, n_features), name="sequence_input")
    ent_in = Input(shape=(1,), name="entity_input")

    lstm_out = LSTM(LSTM_UNITS, activation="tanh")(seq_in)
    emb_layer = Embedding(n_cities, EMBEDDING_DIM, name="entity_embedding")
    emb = Flatten()(emb_layer(ent_in))
    hidden = Dense(SHARED_DENSE_UNITS, activation="relu")(Concatenate()([lstm_out, emb]))
    heads = [Dense(1, name=f"out_{t}")(hidden) for t in TARGET_VARS]

    model = Model([seq_in, ent_in], heads)
    model.compile(optimizer="adam", loss="mse")
    return model, emb_layer


def pretrain_entity_embeddings(X_fit, ent_fit, y_fit, X_val, ent_val, y_val,
                                n_features, n_cities, use_val, label):
    """Train the embedding net on TRAINING years only, then return a fixed
    lookup array embeddings[pid_idx] -> (EMBEDDING_DIM,) vector. No
    forecast leaves this function -- it exists only to harvest the
    Embedding layer's weights."""
    print(f"  Pretraining entity embeddings for '{label}' feature set "
          f"({n_cities} cities, dim={EMBEDDING_DIM})...")
    model, emb_layer = build_embedding_net(n_features, X_fit.shape[1], n_cities)
    monitor = "val_loss" if use_val else "loss"
    stopper = EarlyStopping(monitor=monitor, patience=EMB_PATIENCE, restore_best_weights=True)
    fit_kwargs = dict(epochs=EMB_EPOCHS, batch_size=EMB_BATCH_SIZE, verbose=0, callbacks=[stopper])
    if use_val:
        fit_kwargs["validation_data"] = ([X_val, ent_val], [y_val[:, j] for j in range(y_fit.shape[1])])
    model.fit([X_fit, ent_fit], [y_fit[:, j] for j in range(y_fit.shape[1])], **fit_kwargs)
    return emb_layer.get_weights()[0]   # (n_cities, EMBEDDING_DIM)


def attach_embeddings(X_flat, ent_ids, embeddings):
    """Concatenate each row's city embedding vector onto its flattened
    window features."""
    return np.hstack([X_flat, embeddings[ent_ids]])

# ============================================================= MODELS =====

def fit_predict_mtl_xgb(X_fit, y_fit, X_val, y_val, X_te, use_val):
    """One booster, vector-leaf trees (multi_strategy='multi_output_tree'):
    all four targets fit jointly, hard-parameter-sharing analog for trees."""
    model = xgb.XGBRegressor(multi_strategy="multi_output_tree",
                              n_estimators=XGB_N_ESTIMATORS,
                              early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS if use_val else None,
                              **XGB_PARAMS)
    fit_kwargs = dict(verbose=False)
    if use_val:
        fit_kwargs["eval_set"] = [(X_val, y_val)]
    model.fit(X_fit, y_fit, **fit_kwargs)
    return model.predict(X_te)   # (n, 4)


def fit_predict_stl_xgb(X_fit, y_fit, X_val, y_val, X_te, use_val, n_targets):
    """Four independent boosters (default multi_strategy='one_output_per_tree'),
    one per target -- the tree analog of four independent STL networks."""
    preds = np.zeros((X_te.shape[0], n_targets))
    for j in range(n_targets):
        model = xgb.XGBRegressor(n_estimators=XGB_N_ESTIMATORS,
                                  early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS if use_val else None,
                                  **XGB_PARAMS)
        fit_kwargs = dict(verbose=False)
        if use_val:
            fit_kwargs["eval_set"] = [(X_val, y_val[:, j])]
        model.fit(X_fit, y_fit[:, j], **fit_kwargs)
        preds[:, j] = model.predict(X_te)
    return preds


def _knn_medae_for_k(k, X_fit, y_fit_col, X_val, y_val_col):
    knn = KNeighborsRegressor(n_neighbors=min(k, len(X_fit)), weights=KNN_WEIGHTS)
    knn.fit(X_fit, y_fit_col)
    pred = knn.predict(X_val)
    return float(np.median(np.abs(y_val_col - pred)))


def fit_predict_mtl_knn(X_fit, y_fit, X_val, y_val, X_te, use_val, n_targets):
    """ONE k, chosen to minimize *average* validation MedAE across all four
    targets, feeding a single multi-output KNeighborsRegressor call -- one
    shared neighbor set/representation used for every target (see docstring
    point 3 for why this, not independently-learned weights, is what
    'shared' can mean for KNN)."""
    if use_val and len(X_val) > 0:
        scores = []
        for k in KNN_K_GRID:
            knn = KNeighborsRegressor(n_neighbors=min(k, len(X_fit)), weights=KNN_WEIGHTS)
            knn.fit(X_fit, y_fit)
            pred = knn.predict(X_val)
            scores.append(float(np.median(np.abs(y_val - pred))))   # avg over targets, elementwise median
        best_k = KNN_K_GRID[int(np.argmin(scores))]
    else:
        best_k = KNN_K_GRID[len(KNN_K_GRID) // 2]
    X_train_full = np.vstack([X_fit, X_val]) if use_val and len(X_val) > 0 else X_fit
    y_train_full = np.vstack([y_fit, y_val]) if use_val and len(X_val) > 0 else y_fit
    knn = KNeighborsRegressor(n_neighbors=min(best_k, len(X_train_full)), weights=KNN_WEIGHTS)
    knn.fit(X_train_full, y_train_full)
    print(f"    MTL-KNN shared k = {best_k}")
    return knn.predict(X_te)


def fit_predict_stl_knn(X_fit, y_fit, X_val, y_val, X_te, use_val, n_targets):
    """Four independent KNeighborsRegressor calls, each with its OWN k
    chosen on validation for that target alone -- the KNN analog of four
    independently-tuned STL networks."""
    preds = np.zeros((X_te.shape[0], n_targets))
    chosen_ks = []
    for j in range(n_targets):
        if use_val and len(X_val) > 0:
            scores = [_knn_medae_for_k(k, X_fit, y_fit[:, j], X_val, y_val[:, j]) for k in KNN_K_GRID]
            best_k = KNN_K_GRID[int(np.argmin(scores))]
        else:
            best_k = KNN_K_GRID[len(KNN_K_GRID) // 2]
        chosen_ks.append(best_k)
        X_train_full = np.vstack([X_fit, X_val]) if use_val and len(X_val) > 0 else X_fit
        y_train_full = np.concatenate([y_fit[:, j], y_val[:, j]]) if use_val and len(X_val) > 0 else y_fit[:, j]
        knn = KNeighborsRegressor(n_neighbors=min(best_k, len(X_train_full)), weights=KNN_WEIGHTS)
        knn.fit(X_train_full, y_train_full)
        preds[:, j] = knn.predict(X_te)
    print(f"    STL-KNN per-target k = {dict(zip(TARGET_VARS, chosen_ks))}")
    return preds

# ======================================================= ARIMA BASELINE ===
# Unchanged from the LSTM script.

def arima_forecast_all(df, target_var):
    forecasts = {}
    train = df[df[YEAR_COL] <= TRAIN_END_YEAR]
    for pid, g in train.groupby(ENTITY_COL):
        series = g.sort_values(YEAR_COL)[target_var].astype(float).values
        try:
            fitted = ARIMA(series, order=(1, 1, 0)).fit()
            pred = float(np.asarray(fitted.forecast(steps=1))[0])
        except Exception:
            pred = float(series[-1])
        forecasts[pid] = pred
    return forecasts

# ============================================================= METRICS ====
# Unchanged from the LSTM script.

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(y_true) + np.abs(y_pred)
    out = np.zeros_like(y_true)
    mask = denom != 0
    out[mask] = 2.0 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]
    return float(np.mean(out)) * 100


def med_ae(y_true, y_pred):
    return float(np.median(np.abs(y_true - y_pred)))


def med_smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(y_true) + np.abs(y_pred)
    out = np.zeros_like(y_true)
    mask = denom != 0
    out[mask] = 2.0 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]
    return float(np.median(out)) * 100


# =============================================================== MAIN =====

df = load_data(DATA_PATH)
n_total_cities = df[ENTITY_COL].nunique()

internal_cities = eligible_cities(df, TARGET_VARS)
full_cities = eligible_cities(df, TARGET_VARS + EXTERNAL_VARS)
print(f"{len(internal_cities)}/{n_total_cities} cities have complete target "
      f"data -> used by ARIMA, MTL/STL-*-internal")
print(f"{len(full_cities)}/{n_total_cities} cities have complete target+external "
      f"data -> used by MTL/STL-*-full")

n_targets = len(TARGET_VARS)
results = []

# ---- ARIMA baseline (complete-target cities only) ----
df_arima = df[df[ENTITY_COL].isin(internal_cities)]
print(f"\nFitting ARIMA baselines on {len(internal_cities)} cities...")
arima_by_target = {t: arima_forecast_all(df_arima, t) for t in TARGET_VARS}
test_rows = df_arima[df_arima[YEAR_COL] == FORECAST_YEAR].set_index(ENTITY_COL)

for target in TARGET_VARS:
    y_true = test_rows[target].values
    y_pred = np.array([arima_by_target[target].get(pid, np.nan) for pid in test_rows.index])
    valid = ~np.isnan(y_pred)
    results.append({"target_variable": target, "model": "ARIMA",
                     "n_cities_used": len(internal_cities),
                     "n_test_cities": int(valid.sum()),
                     "MedAE": round(med_ae(y_true[valid], y_pred[valid]), 2),
                     "sMdAPE_%": round(med_smape(y_true[valid], y_pred[valid]), 2)})

# ---- XGBoost + KNN runs: {full, internal} feature sets x {MTL, STL} ----
for feature_label, feature_cols in FEATURE_SETS.items():
    cities = full_cities if feature_label == "full" else internal_cities
    n_cities = len(cities)
    print(f"\nRunning tree/KNN models on '{feature_label}' features ({n_cities} cities)...")

    df_sub = df[df[ENTITY_COL].isin(cities)].copy()
    pid_to_idx = {pid: i for i, pid in enumerate(cities)}
    df_sub[ENTITY_COL] = df_sub[ENTITY_COL].map(pid_to_idx)

    X_seq, X_ent, y, year = build_sequences(df_sub, feature_cols, WINDOW_SIZE)
    train_mask, test_mask = year <= TRAIN_END_YEAR, year == FORECAST_YEAR

    x_scalers = fit_city_scalers(df_sub, feature_cols)
    y_scalers = fit_city_scalers(df_sub, TARGET_VARS)
    X_tr = apply_scaler(X_seq[train_mask], X_ent[train_mask], x_scalers, is_sequence=True)
    X_te = apply_scaler(X_seq[test_mask], X_ent[test_mask], x_scalers, is_sequence=True)
    y_tr = apply_scaler(y[train_mask], X_ent[train_mask], y_scalers, is_sequence=False)
    ent_tr, ent_te, y_te = X_ent[train_mask], X_ent[test_mask], y[test_mask]
    year_tr = year[train_mask]
    n_features = len(feature_cols)

    use_val = USE_VALIDATION and VAL_YEARS > 0
    if use_val:
        fit_sel = year_tr <= (TRAIN_END_YEAR - VAL_YEARS)
        val_sel = ~fit_sel
    else:
        fit_sel = np.ones_like(year_tr, dtype=bool)
        val_sel = np.zeros_like(year_tr, dtype=bool)

    X_fit_seq, ent_fit, y_fit = X_tr[fit_sel], ent_tr[fit_sel], y_tr[fit_sel]
    X_val_seq, ent_val, y_val = X_tr[val_sel], ent_tr[val_sel], y_tr[val_sel]
    if use_val and len(y_val) == 0:
        print(f"  [warning] VAL_YEARS={VAL_YEARS} left no validation rows for "
              f"'{feature_label}' -- falling back to no early stopping / mid-grid k.")
        use_val = False

    # --- 1. pretrain entity embeddings on TRAINING years only ---
    embeddings = pretrain_entity_embeddings(
        X_fit_seq, ent_fit, y_fit, X_val_seq, ent_val, y_val,
        n_features, n_cities, use_val, feature_label)

    # --- 2. flatten windows + attach the fixed per-city embedding vector ---
    X_fit = attach_embeddings(flatten_sequences(X_fit_seq), ent_fit, embeddings)
    X_val = attach_embeddings(flatten_sequences(X_val_seq), ent_val, embeddings) if use_val else np.empty((0, X_fit.shape[1]))
    X_te_flat = attach_embeddings(flatten_sequences(X_te), ent_te, embeddings)

    # --- 3. XGBoost ---
    mtl_xgb_pred = fit_predict_mtl_xgb(X_fit, y_fit, X_val, y_val, X_te_flat, use_val)
    mtl_xgb_pred = inverse_full(mtl_xgb_pred, ent_te, y_scalers)
    stl_xgb_pred = fit_predict_stl_xgb(X_fit, y_fit, X_val, y_val, X_te_flat, use_val, n_targets)
    stl_xgb_pred_inv = np.column_stack([
        inverse_col(stl_xgb_pred[:, j], ent_te, y_scalers, j) for j in range(n_targets)
    ])

    # --- 4. KNN ---
    mtl_knn_pred = fit_predict_mtl_knn(X_fit, y_fit, X_val, y_val, X_te_flat, use_val, n_targets)
    mtl_knn_pred = inverse_full(mtl_knn_pred, ent_te, y_scalers)
    stl_knn_pred = fit_predict_stl_knn(X_fit, y_fit, X_val, y_val, X_te_flat, use_val, n_targets)
    stl_knn_pred_inv = np.column_stack([
        inverse_col(stl_knn_pred[:, j], ent_te, y_scalers, j) for j in range(n_targets)
    ])

    for j, target in enumerate(TARGET_VARS):
        y_true = y_te[:, j]
        for model_name, y_pred in [
            (f"MTL-XGB ({feature_label})", mtl_xgb_pred[:, j]),
            (f"STL-XGB ({feature_label})", stl_xgb_pred_inv[:, j]),
            (f"MTL-KNN ({feature_label})", mtl_knn_pred[:, j]),
            (f"STL-KNN ({feature_label})", stl_knn_pred_inv[:, j]),
        ]:
            valid = ~np.isnan(y_pred)
            results.append({"target_variable": target, "model": model_name,
                             "n_cities_used": n_cities,
                             "n_test_cities": int(valid.sum()),
                             "MedAE": round(med_ae(y_true[valid], y_pred[valid]), 2),
                             "sMdAPE_%": round(med_smape(y_true[valid], y_pred[valid]), 2)})

results_df = pd.DataFrame(results)
print(f"\nDone. {len(results_df)} rows stored in `results_df`. Run the report cell next.")


# ===========================================================  REPORT ===
pd.set_option("display.width", 120)

MODEL_ORDER_NOTE = ("Note: ARIMA / *-internal models and *-full models are NOT "
                     "scored on the same set of cities -- see n_cities_used row. "
                     "See the file header for what MTL/STL and 'entity embedded' "
                     "specifically mean for XGBoost vs. KNN -- they are not the "
                     "same guarantee the LSTM script's MTL/STL split gives you.")

MODEL_ORDER = ["ARIMA",
               "MTL-XGB (internal)", "STL-XGB (internal)",
               "MTL-XGB (full)", "STL-XGB (full)",
               "MTL-KNN (internal)", "STL-KNN (internal)",
               "MTL-KNN (full)", "STL-KNN (full)"]

HYPOTHESES = [
    ("H1", "MTL-XGB (full) < ARIMA",              "MTL-XGB (full)", "ARIMA"),
    ("H2", "MTL-XGB (full) < STL-XGB (full)",     "MTL-XGB (full)", "STL-XGB (full)"),
    ("H3", "MTL-XGB (full) < MTL-XGB (internal)", "MTL-XGB (full)", "MTL-XGB (internal)"),
    ("H4", "STL-XGB (full) < STL-XGB (internal)", "STL-XGB (full)", "STL-XGB (internal)"),
    ("H5", "MTL-KNN (full) < ARIMA",              "MTL-KNN (full)", "ARIMA"),
    ("H6", "MTL-KNN (full) < STL-KNN (full)",     "MTL-KNN (full)", "STL-KNN (full)"),
    ("H7", "MTL-KNN (full) < MTL-KNN (internal)", "MTL-KNN (full)", "MTL-KNN (internal)"),
    ("H8", "STL-KNN (full) < STL-KNN (internal)", "STL-KNN (full)", "STL-KNN (internal)"),
    ("H9", "MTL-XGB (full) < MTL-KNN (full)",     "MTL-XGB (full)", "MTL-KNN (full)"),
]

overview_rows = []

for target in TARGET_VARS:
    sub = results_df[results_df["target_variable"] == target].set_index("model")
    sub = sub[["n_cities_used", "n_test_cities", "MedAE", "sMdAPE_%"]]
    sub = sub.reindex([m for m in MODEL_ORDER if m in sub.index])

    best_model = sub["MedAE"].idxmin()
    print(f"\n=== {target} (best: {best_model}) ===")
    print(sub.T.to_string())

    overview_row = {"target_variable": target}
    for h_id, label, model_a, model_b in HYPOTHESES:
        if model_a in sub.index and model_b in sub.index:
            supported = bool(sub.loc[model_a, "MedAE"] < sub.loc[model_b, "MedAE"])
            print(f"  {h_id} ({label}): "
                  f"{'SUPPORTED' if supported else 'NOT SUPPORTED'} "
                  f"[{sub.loc[model_a, 'MedAE']:.2f} vs {sub.loc[model_b, 'MedAE']:.2f}]")
            overview_row[h_id] = "Y" if supported else "N"
        else:
            print(f"  {h_id} ({label}): N/A (missing model rows)")
            overview_row[h_id] = "N/A"
    overview_rows.append(overview_row)

overview_df = pd.DataFrame(overview_rows).set_index("target_variable")
print("\n=== Hypothesis overview (Y = supported, N = not supported, per target) ===")
print(overview_df.to_string())
print(f"\n{MODEL_ORDER_NOTE}")

258/274 cities have complete target data -> used by ARIMA, MTL/STL-*-internal
257/274 cities have complete target+external data -> used by MTL/STL-*-full

Fitting ARIMA baselines on 258 cities...

Running tree/KNN models on 'full' features (257 cities)...
  Pretraining entity embeddings for 'full' feature set (257 cities, dim=4)...
    MTL-KNN shared k = 3
    STL-KNN per-target k = {'opr_ratio_gn': 20, 'opr_ratio_ep': 5, 'cash_ratio_totasst': 3, 'totdebt_to_asst': 3, 'capital_to_asst': 7, 'funded_ratio_total': 3}

Running tree/KNN models on 'internal' features (258 cities)...
  Pretraining entity embeddings for 'internal' feature set (258 cities, dim=4)...
    MTL-KNN shared k = 5
    STL-KNN per-target k = {'opr_ratio_gn': 5, 'opr_ratio_ep': 10, 'cash_ratio_totasst': 5, 'totdebt_to_asst': 15, 'capital_to_asst': 3, 'funded_ratio_total': 5}

Done. 54 rows stored in `results_df`. Run the report cell next.

=== opr_ratio_gn (best: STL-XGB (internal)) ===
model           ARIMA  MTL-XGB (i

In [2]:
"""
Same XGBoost / KNN forecasting comparison as the entity-embedding version,
but WITHOUT the entity-embedding pretraining step. Flattened window
features are fed to XGBoost/KNN directly (no per-city embedding vector
concatenated on). Everything else -- config, data prep, city scaling,
train/val split, ARIMA baseline, MTL/STL definitions for XGB and KNN,
metrics, and the hypothesis report -- is unchanged.

Per-city errors are computed and MEDIANS (MedAE / sMdAPE) are used for
all comparisons, exactly as before.
"""

# ============================================================== CONFIG ====
DATA_PATH = "variable.xlsx"
TARGET_VARS = ["cur_bal_gn", "cash_ratio_totasst","una_ga","una_ba", "nic", "funded_ratio_total"]
EXTERNAL_VARS = ["ln_pop_city", "ln_curgdp", "ln_psnl_incm", "employment", "ln_med_homevalue",
                    "property_rel", "intg_rev_rel",
                    "disaster_event", "flood_dmg_tot",
                    "under_18%", "age_65_plus%", "white%", "bachelors%", "disability%", "poverty%"
                    ]
ENTITY_COL, YEAR_COL = "pid", "year"

WINDOW_SIZE = 3           # years of history each model sees before predicting
TRAIN_END_YEAR = 2023     # last year usable as a TRAINING target
FORECAST_YEAR = 2024      # the held-out year to score against
# One-step-ahead only: the model's input window is the WINDOW_SIZE actual
# years immediately before FORECAST_YEAR.

# Validation split: used for (a) XGBoost early stopping, and (b) choosing k
# for both KNN variants. The most recent VAL_YEARS training years are held
# out from fitting and used only to pick when-to-stop / which-k.
USE_VALIDATION = True
VAL_YEARS = 1

# --- XGBoost ---
XGB_N_ESTIMATORS = 500
XGB_EARLY_STOPPING_ROUNDS = 20
XGB_PARAMS = dict(tree_method="hist", max_depth=3, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, eval_metric="rmse",
                   random_state=42)

# --- KNN ---
KNN_K_GRID = [3, 5, 7, 10, 15, 20, 30]   # candidate neighbor counts to try
KNN_WEIGHTS = "distance"                  # closer neighbors count more

FEATURE_SETS = {
    "full": TARGET_VARS + EXTERNAL_VARS,
    "internal": TARGET_VARS,
}

# =========================================================== DATA PREP ====
# (unchanged)

def load_data(path):
    """Read Excel and keep only the needed columns. No filling/imputation
    happens here or anywhere else in this script."""
    df = pd.read_excel(path)
    keep = [ENTITY_COL, "city", YEAR_COL] + TARGET_VARS + EXTERNAL_VARS
    return df[keep].sort_values([ENTITY_COL, YEAR_COL]).reset_index(drop=True)


def eligible_cities(df, cols):
    """Cities with NO missing value in any of `cols`, across every year
    they appear. Returns a sorted list of city ids (original, un-recoded)."""
    incomplete = df.loc[df[cols].isnull().any(axis=1), ENTITY_COL].unique()
    return sorted(set(df[ENTITY_COL].unique()) - set(incomplete))


def build_sequences(df, feature_cols, window_size):
    """Slide a window of `window_size` years per city.
    Returns X_seq (n, window, n_features), X_ent (n,), y (n, 4 targets),
    target_year (n,)."""
    X_seq, X_ent, y, target_year = [], [], [], []
    for pid, g in df.groupby(ENTITY_COL):
        g = g.sort_values(YEAR_COL)
        feats = g[feature_cols].values
        targs = g[TARGET_VARS].values
        years = g[YEAR_COL].values
        for i in range(len(g) - window_size):
            X_seq.append(feats[i:i + window_size])
            X_ent.append(pid)
            y.append(targs[i + window_size])
            target_year.append(years[i + window_size])
    return (np.array(X_seq, dtype="float32"), np.array(X_ent, dtype="int32"),
            np.array(y, dtype="float32"), np.array(target_year))


def flatten_sequences(X_seq):
    """(n, window, n_features) -> (n, window*n_features). XGBoost and KNN
    take flat feature vectors, not sequences, so the window is unrolled
    into columns (year0_var0, year0_var1, ..., year(w-1)_var(k-1))."""
    n = X_seq.shape[0]
    return X_seq.reshape(n, -1)

# ======================================================= CITY SCALING =====
# Unchanged: one MinMaxScaler per city per role (features vs. targets),
# fit on 2013-2023 only, inverted with that same city's parameters before
# scoring.

def fit_city_scalers(df, cols):
    train = df[df[YEAR_COL] <= TRAIN_END_YEAR]
    return {pid: MinMaxScaler().fit(g[cols].values) for pid, g in train.groupby(ENTITY_COL)}


def apply_scaler(arr, ent_ids, scalers, is_sequence):
    out = np.zeros_like(arr, dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        scaler = scalers[pid]
        if is_sequence:
            n, w, c = arr[mask].shape
            out[mask] = scaler.transform(arr[mask].reshape(-1, c)).reshape(n, w, c)
        else:
            out[mask] = scaler.transform(arr[mask])
    return out


def inverse_full(y_scaled, ent_ids, scalers):
    out = np.zeros_like(y_scaled, dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        out[mask] = scalers[pid].inverse_transform(y_scaled[mask])
    return out


def inverse_col(y_scaled_col, ent_ids, scalers, col_index):
    out = np.zeros(len(y_scaled_col), dtype=float)
    for pid in np.unique(ent_ids):
        mask = ent_ids == pid
        scaler = scalers[pid]
        dummy = np.zeros((int(mask.sum()), scaler.n_features_in_))
        dummy[:, col_index] = y_scaled_col[mask]
        out[mask] = scaler.inverse_transform(dummy)[:, col_index]
    return out

# ============================================================= MODELS =====
# No entity-embedding step in this version: XGBoost/KNN consume the
# flattened window features directly (attach_embeddings is gone; X_fit /
# X_val / X_te are just flatten_sequences(...) with no concatenation).

def fit_predict_mtl_xgb(X_fit, y_fit, X_val, y_val, X_te, use_val):
    """One booster, vector-leaf trees (multi_strategy='multi_output_tree'):
    all four targets fit jointly, hard-parameter-sharing analog for trees."""
    model = xgb.XGBRegressor(multi_strategy="multi_output_tree",
                              n_estimators=XGB_N_ESTIMATORS,
                              early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS if use_val else None,
                              **XGB_PARAMS)
    fit_kwargs = dict(verbose=False)
    if use_val:
        fit_kwargs["eval_set"] = [(X_val, y_val)]
    model.fit(X_fit, y_fit, **fit_kwargs)
    return model.predict(X_te)   # (n, 4)


def fit_predict_stl_xgb(X_fit, y_fit, X_val, y_val, X_te, use_val, n_targets):
    """Four independent boosters (default multi_strategy='one_output_per_tree'),
    one per target -- the tree analog of four independent STL networks."""
    preds = np.zeros((X_te.shape[0], n_targets))
    for j in range(n_targets):
        model = xgb.XGBRegressor(n_estimators=XGB_N_ESTIMATORS,
                                  early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS if use_val else None,
                                  **XGB_PARAMS)
        fit_kwargs = dict(verbose=False)
        if use_val:
            fit_kwargs["eval_set"] = [(X_val, y_val[:, j])]
        model.fit(X_fit, y_fit[:, j], **fit_kwargs)
        preds[:, j] = model.predict(X_te)
    return preds


def _knn_medae_for_k(k, X_fit, y_fit_col, X_val, y_val_col):
    knn = KNeighborsRegressor(n_neighbors=min(k, len(X_fit)), weights=KNN_WEIGHTS)
    knn.fit(X_fit, y_fit_col)
    pred = knn.predict(X_val)
    return float(np.median(np.abs(y_val_col - pred)))


def fit_predict_mtl_knn(X_fit, y_fit, X_val, y_val, X_te, use_val, n_targets):
    """ONE k, chosen to minimize *average* validation MedAE across all four
    targets, feeding a single multi-output KNeighborsRegressor call -- one
    shared neighbor set/representation used for every target."""
    if use_val and len(X_val) > 0:
        scores = []
        for k in KNN_K_GRID:
            knn = KNeighborsRegressor(n_neighbors=min(k, len(X_fit)), weights=KNN_WEIGHTS)
            knn.fit(X_fit, y_fit)
            pred = knn.predict(X_val)
            scores.append(float(np.median(np.abs(y_val - pred))))   # avg over targets, elementwise median
        best_k = KNN_K_GRID[int(np.argmin(scores))]
    else:
        best_k = KNN_K_GRID[len(KNN_K_GRID) // 2]
    X_train_full = np.vstack([X_fit, X_val]) if use_val and len(X_val) > 0 else X_fit
    y_train_full = np.vstack([y_fit, y_val]) if use_val and len(X_val) > 0 else y_fit
    knn = KNeighborsRegressor(n_neighbors=min(best_k, len(X_train_full)), weights=KNN_WEIGHTS)
    knn.fit(X_train_full, y_train_full)
    print(f"    MTL-KNN shared k = {best_k}")
    return knn.predict(X_te)


def fit_predict_stl_knn(X_fit, y_fit, X_val, y_val, X_te, use_val, n_targets):
    """Four independent KNeighborsRegressor calls, each with its OWN k
    chosen on validation for that target alone -- the KNN analog of four
    independently-tuned STL networks."""
    preds = np.zeros((X_te.shape[0], n_targets))
    chosen_ks = []
    for j in range(n_targets):
        if use_val and len(X_val) > 0:
            scores = [_knn_medae_for_k(k, X_fit, y_fit[:, j], X_val, y_val[:, j]) for k in KNN_K_GRID]
            best_k = KNN_K_GRID[int(np.argmin(scores))]
        else:
            best_k = KNN_K_GRID[len(KNN_K_GRID) // 2]
        chosen_ks.append(best_k)
        X_train_full = np.vstack([X_fit, X_val]) if use_val and len(X_val) > 0 else X_fit
        y_train_full = np.concatenate([y_fit[:, j], y_val[:, j]]) if use_val and len(X_val) > 0 else y_fit[:, j]
        knn = KNeighborsRegressor(n_neighbors=min(best_k, len(X_train_full)), weights=KNN_WEIGHTS)
        knn.fit(X_train_full, y_train_full)
        preds[:, j] = knn.predict(X_te)
    print(f"    STL-KNN per-target k = {dict(zip(TARGET_VARS, chosen_ks))}")
    return preds

# ======================================================= ARIMA BASELINE ===
# Unchanged.

def arima_forecast_all(df, target_var):
    forecasts = {}
    train = df[df[YEAR_COL] <= TRAIN_END_YEAR]
    for pid, g in train.groupby(ENTITY_COL):
        series = g.sort_values(YEAR_COL)[target_var].astype(float).values
        try:
            fitted = ARIMA(series, order=(1, 1, 0)).fit()
            pred = float(np.asarray(fitted.forecast(steps=1))[0])
        except Exception:
            pred = float(series[-1])
        forecasts[pid] = pred
    return forecasts

# ============================================================= METRICS ====
# Unchanged. MAE/sMAPE kept for completeness even though only the
# median-based versions (MedAE / sMdAPE) are used in the report.

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))


def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(y_true) + np.abs(y_pred)
    out = np.zeros_like(y_true)
    mask = denom != 0
    out[mask] = 2.0 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]
    return float(np.mean(out)) * 100


def med_ae(y_true, y_pred):
    """Per-city absolute errors -> median (MedAE)."""
    return float(np.median(np.abs(y_true - y_pred)))


def med_smape(y_true, y_pred):
    """Per-city sMAPE -> median (sMdAPE)."""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.abs(y_true) + np.abs(y_pred)
    out = np.zeros_like(y_true)
    mask = denom != 0
    out[mask] = 2.0 * np.abs(y_true[mask] - y_pred[mask]) / denom[mask]
    return float(np.median(out)) * 100


# =============================================================== MAIN =====

df = load_data(DATA_PATH)
n_total_cities = df[ENTITY_COL].nunique()

internal_cities = eligible_cities(df, TARGET_VARS)
full_cities = eligible_cities(df, TARGET_VARS + EXTERNAL_VARS)
print(f"{len(internal_cities)}/{n_total_cities} cities have complete target "
      f"data -> used by ARIMA, MTL/STL-*-internal")
print(f"{len(full_cities)}/{n_total_cities} cities have complete target+external "
      f"data -> used by MTL/STL-*-full")

n_targets = len(TARGET_VARS)
results = []

# ---- ARIMA baseline (complete-target cities only) ----
df_arima = df[df[ENTITY_COL].isin(internal_cities)]
print(f"\nFitting ARIMA baselines on {len(internal_cities)} cities...")
arima_by_target = {t: arima_forecast_all(df_arima, t) for t in TARGET_VARS}
test_rows = df_arima[df_arima[YEAR_COL] == FORECAST_YEAR].set_index(ENTITY_COL)

for target in TARGET_VARS:
    y_true = test_rows[target].values
    y_pred = np.array([arima_by_target[target].get(pid, np.nan) for pid in test_rows.index])
    valid = ~np.isnan(y_pred)
    results.append({"target_variable": target, "model": "ARIMA",
                     "n_cities_used": len(internal_cities),
                     "n_test_cities": int(valid.sum()),
                     "MedAE": round(med_ae(y_true[valid], y_pred[valid]), 2),
                     "sMdAPE_%": round(med_smape(y_true[valid], y_pred[valid]), 2)})

# ---- XGBoost + KNN runs: {full, internal} feature sets x {MTL, STL} ----
for feature_label, feature_cols in FEATURE_SETS.items():
    cities = full_cities if feature_label == "full" else internal_cities
    n_cities = len(cities)
    print(f"\nRunning tree/KNN models on '{feature_label}' features ({n_cities} cities)...")

    df_sub = df[df[ENTITY_COL].isin(cities)].copy()
    pid_to_idx = {pid: i for i, pid in enumerate(cities)}
    df_sub[ENTITY_COL] = df_sub[ENTITY_COL].map(pid_to_idx)

    X_seq, X_ent, y, year = build_sequences(df_sub, feature_cols, WINDOW_SIZE)
    train_mask, test_mask = year <= TRAIN_END_YEAR, year == FORECAST_YEAR

    x_scalers = fit_city_scalers(df_sub, feature_cols)
    y_scalers = fit_city_scalers(df_sub, TARGET_VARS)
    X_tr = apply_scaler(X_seq[train_mask], X_ent[train_mask], x_scalers, is_sequence=True)
    X_te = apply_scaler(X_seq[test_mask], X_ent[test_mask], x_scalers, is_sequence=True)
    y_tr = apply_scaler(y[train_mask], X_ent[train_mask], y_scalers, is_sequence=False)
    ent_tr, ent_te, y_te = X_ent[train_mask], X_ent[test_mask], y[test_mask]
    year_tr = year[train_mask]

    use_val = USE_VALIDATION and VAL_YEARS > 0
    if use_val:
        fit_sel = year_tr <= (TRAIN_END_YEAR - VAL_YEARS)
        val_sel = ~fit_sel
    else:
        fit_sel = np.ones_like(year_tr, dtype=bool)
        val_sel = np.zeros_like(year_tr, dtype=bool)

    X_fit_seq, ent_fit, y_fit = X_tr[fit_sel], ent_tr[fit_sel], y_tr[fit_sel]
    X_val_seq, ent_val, y_val = X_tr[val_sel], ent_tr[val_sel], y_tr[val_sel]
    if use_val and len(y_val) == 0:
        print(f"  [warning] VAL_YEARS={VAL_YEARS} left no validation rows for "
              f"'{feature_label}' -- falling back to no early stopping / mid-grid k.")
        use_val = False

    # --- flatten windows directly, no entity embedding attached ---
    X_fit = flatten_sequences(X_fit_seq)
    X_val = flatten_sequences(X_val_seq) if use_val else np.empty((0, X_fit.shape[1]))
    X_te_flat = flatten_sequences(X_te)

    # --- XGBoost ---
    mtl_xgb_pred = fit_predict_mtl_xgb(X_fit, y_fit, X_val, y_val, X_te_flat, use_val)
    mtl_xgb_pred = inverse_full(mtl_xgb_pred, ent_te, y_scalers)
    stl_xgb_pred = fit_predict_stl_xgb(X_fit, y_fit, X_val, y_val, X_te_flat, use_val, n_targets)
    stl_xgb_pred_inv = np.column_stack([
        inverse_col(stl_xgb_pred[:, j], ent_te, y_scalers, j) for j in range(n_targets)
    ])

    # --- KNN ---
    mtl_knn_pred = fit_predict_mtl_knn(X_fit, y_fit, X_val, y_val, X_te_flat, use_val, n_targets)
    mtl_knn_pred = inverse_full(mtl_knn_pred, ent_te, y_scalers)
    stl_knn_pred = fit_predict_stl_knn(X_fit, y_fit, X_val, y_val, X_te_flat, use_val, n_targets)
    stl_knn_pred_inv = np.column_stack([
        inverse_col(stl_knn_pred[:, j], ent_te, y_scalers, j) for j in range(n_targets)
    ])

    for j, target in enumerate(TARGET_VARS):
        y_true = y_te[:, j]
        for model_name, y_pred in [
            (f"MTL-XGB ({feature_label})", mtl_xgb_pred[:, j]),
            (f"STL-XGB ({feature_label})", stl_xgb_pred_inv[:, j]),
            (f"MTL-KNN ({feature_label})", mtl_knn_pred[:, j]),
            (f"STL-KNN ({feature_label})", stl_knn_pred_inv[:, j]),
        ]:
            valid = ~np.isnan(y_pred)
            results.append({"target_variable": target, "model": model_name,
                             "n_cities_used": n_cities,
                             "n_test_cities": int(valid.sum()),
                             "MedAE": round(med_ae(y_true[valid], y_pred[valid]), 2),
                             "sMdAPE_%": round(med_smape(y_true[valid], y_pred[valid]), 2)})

results_df = pd.DataFrame(results)
print(f"\nDone. {len(results_df)} rows stored in `results_df`. Run the report cell next.")


# ===========================================================  REPORT ===
pd.set_option("display.width", 120)

MODEL_ORDER_NOTE = ("Note: ARIMA / *-internal models and *-full models are NOT "
                     "scored on the same set of cities -- see n_cities_used row. "
                     "This version has no entity-embedding step, so XGBoost/KNN "
                     "see raw flattened window features only (no per-city vector). "
                     "See the file header for what MTL/STL specifically mean for "
                     "XGBoost vs. KNN.")

MODEL_ORDER = ["ARIMA",
               "MTL-XGB (internal)", "STL-XGB (internal)",
               "MTL-XGB (full)", "STL-XGB (full)",
               "MTL-KNN (internal)", "STL-KNN (internal)",
               "MTL-KNN (full)", "STL-KNN (full)"]

HYPOTHESES = [
    ("H1", "MTL-XGB (full) < ARIMA",              "MTL-XGB (full)", "ARIMA"),
    ("H2", "MTL-XGB (full) < STL-XGB (full)",     "MTL-XGB (full)", "STL-XGB (full)"),
    ("H3", "MTL-XGB (full) < MTL-XGB (internal)", "MTL-XGB (full)", "MTL-XGB (internal)"),
    ("H4", "STL-XGB (full) < STL-XGB (internal)", "STL-XGB (full)", "STL-XGB (internal)"),
    ("H5", "MTL-KNN (full) < ARIMA",              "MTL-KNN (full)", "ARIMA"),
    ("H6", "MTL-KNN (full) < STL-KNN (full)",     "MTL-KNN (full)", "STL-KNN (full)"),
    ("H7", "MTL-KNN (full) < MTL-KNN (internal)", "MTL-KNN (full)", "MTL-KNN (internal)"),
    ("H8", "STL-KNN (full) < STL-KNN (internal)", "STL-KNN (full)", "STL-KNN (internal)"),
    ("H9", "MTL-XGB (full) < MTL-KNN (full)",     "MTL-XGB (full)", "MTL-KNN (full)"),
]

overview_rows = []

for target in TARGET_VARS:
    sub = results_df[results_df["target_variable"] == target].set_index("model")
    sub = sub[["n_cities_used", "n_test_cities", "MedAE", "sMdAPE_%"]]
    sub = sub.reindex([m for m in MODEL_ORDER if m in sub.index])

    best_model = sub["MedAE"].idxmin()
    print(f"\n=== {target} (best: {best_model}) ===")
    print(sub.T.to_string())

    overview_row = {"target_variable": target}
    for h_id, label, model_a, model_b in HYPOTHESES:
        if model_a in sub.index and model_b in sub.index:
            supported = bool(sub.loc[model_a, "MedAE"] < sub.loc[model_b, "MedAE"])
            print(f"  {h_id} ({label}): "
                  f"{'SUPPORTED' if supported else 'NOT SUPPORTED'} "
                  f"[{sub.loc[model_a, 'MedAE']:.2f} vs {sub.loc[model_b, 'MedAE']:.2f}]")
            overview_row[h_id] = "Y" if supported else "N"
        else:
            print(f"  {h_id} ({label}): N/A (missing model rows)")
            overview_row[h_id] = "N/A"
    overview_rows.append(overview_row)

overview_df = pd.DataFrame(overview_rows).set_index("target_variable")
print("\n=== Hypothesis overview (Y = supported, N = not supported, per target) ===")
print(overview_df.to_string())
print(f"\n{MODEL_ORDER_NOTE}")

PermissionError: [Errno 13] Permission denied: 'variable.xlsx'

In [2]:
"""
One-step-ahead forecasting of local-government fiscal ratios.
ARIMA(1,1,0)  vs  {LSTM, XGBoost, KNN} x {MTL, STL} x {internal, full}.

CORE IDEA
--------------------------------------------------------------------
Every ML model predicts the CHANGE in a ratio, delta_y(t) = y(t) - y(t-1),
from the previous WINDOW (=3) years, and the level forecast is rebuilt as

    y_pred(t) = y(t-1) [observed]  +  delta_y_pred(t)

Timeline (levels 2013-2023 -> 10 yearly changes 2014-2023):
  * one sample = predict delta(t) from delta(t-3), delta(t-2), delta(t-1)
    (this needs levels from t-4 to t)
  * target years 2017-2023 -> 7 samples per city
      fit  : target years 2017-2022 (6 per city)
      val  : target year  2023      (1 per city; early stopping / KNN k)
      test : target year  2024      (levels 2020-2023 -> forecast 2024)

INPUT DESIGNS (what each model sees for each of the 3 lagged years)
--------------------------------------------------------------------
  STL-internal : own lagged delta only (purely univariate)
  MTL-internal : lagged deltas of ALL six indicators (no externals)
  STL-full     : own lagged delta + external variables (no other indicators)
  MTL-full     : lagged deltas of ALL six indicators + external variables

  * Lagged indicator deltas are used raw (no scaling), targets are raw deltas.
  * External variables enter as year-over-year LEVEL CHANGES (see
    EXTERNAL_TRANSFORM), lagged over the same 3 years. Zero-heavy shock
    variables (disaster events, flood damage) are NOT differenced; they
    enter as levels.
  * MTL = one model / one shared k predicting all six deltas jointly.
    STL = six independent models, one per indicator.
  * Every ML model gets a city-specific entity embedding (fixed-effect-like):
    - LSTM: native Embedding layer, trained end-to-end.
    - XGBoost / KNN: the embedding vectors harvested from the LSTM trained
      on the SAME design, concatenated onto the flattened window features.
  * Vanilla MSE loss everywhere, no target/input scaling.

EVALUATION
--------------------------------------------------------------------
All models are scored on the SAME cities (complete target + external data),
one test year. Per indicator (models as columns):
  MedAE / sMdAPE_%   median over cities of the absolute / symmetric % error
Hypotheses are judged by median MedAE. A separate table reports each model's
WINNING RATE: the share of cities where that model has the smallest absolute
error among all models (ties split equally).
"""

import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Flatten, Concatenate
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================== CONFIG ====
DATA_PATH = "variable.xlsx"
ENTITY_COL, YEAR_COL = "pid", "year"

TARGET_VARS = ["tot_rev_gn", "tot_exp_gn","cash_ga", "una_ga", "una_ba", "nic", "total_assets_GA", "total_liabilities_GA","tot_fnp"]
# How each external variable enters the model, as a year-over-year LEVEL CHANGE:
#   "diff"    : x_t - x_{t-1}
#                 ln_ variables   -> log growth (unit-free)
#                 shares/percents -> percentage-point change
#   "logdiff" : ln(x_t) - ln(x_{t-1}) for a RAW, strictly positive variable
#               (counts, dollars): unit-free growth instead of size-dependent
#               raw changes. Use this if you import raw values.
#   "level"   : x_t as is -> zero-heavy SHOCK variables (0/1 events, counts).
#               Differencing a shock invents a fake "recovery" signal.
#   "log1p"   : ln(1 + x_t) -> zero-heavy amounts (e.g., flood damage in $).
# >>> CHECK against your variable units; edit freely. <<<
EXTERNAL_TRANSFORM = {
    "ln_pop_city": "diff",
    "ln_curgdp": "diff",
    "ln_psnl_incm": "diff",
    "employment": "logdiff",     # safe for a positive count or rate; use "diff" if
                                 # you prefer a plain level change of a rate
    "ln_med_homevalue": "diff",
    "disaster_event": "level",   # shock: 0/1 (or count) as is
    "flood_dmg_tot": "log1p",    # shock: zero-heavy dollar amount
    "property_rel": "diff",      # percentage-point change
    "intg_rev_rel": "diff",      # percentage-point change
    "under_18%": "diff",
    "age_65_plus%": "diff",
    "white%": "diff",
    "bachelors%": "diff",
    "disability%": "diff",
    "poverty%": "diff",
}
EXTERNAL_VARS = list(EXTERNAL_TRANSFORM)

FIRST_YEAR = 2013         # first year of levels
TRAIN_END_YEAR = 2023     # last year usable as a training target
FORECAST_YEAR = 2024      # held-out year to score
WINDOW = 5                # lagged years of history each model sees
VAL_YEARS = 1             # most recent training target year(s) held out for
                          # early stopping / choosing KNN k

# LSTM (embedding dim = size of the city-specific vector)
EMBEDDING_DIM, LSTM_UNITS, DENSE_UNITS = 4, 16, 16
EPOCHS, BATCH_SIZE, PATIENCE = 200, 32, 15

# XGBoost
XGB_N_ESTIMATORS = 500
XGB_EARLY_STOPPING_ROUNDS = 20
XGB_PARAMS = dict(tree_method="hist", max_depth=3, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8, eval_metric="rmse",
                  random_state=42)

# KNN
KNN_K_GRID = [3, 5, 7, 10, 15, 20, 30]
KNN_WEIGHTS = "distance"

FAMILIES = ["LSTM", "XGB", "KNN"]

# =========================================================== DATA PREP ====

def load_data(path):
    df = pd.read_excel(path)
    keep = [ENTITY_COL, "city", YEAR_COL] + TARGET_VARS + EXTERNAL_VARS
    return df[keep].sort_values([ENTITY_COL, YEAR_COL]).reset_index(drop=True)


def eligible_cities(df, cols):
    """Cities observed in EVERY year FIRST_YEAR..FORECAST_YEAR (no gaps, no
    duplicates) with no missing value in `cols`. Nothing is imputed."""
    need = set(range(FIRST_YEAR, FORECAST_YEAR + 1))
    d = df[(df[YEAR_COL] >= FIRST_YEAR) & (df[YEAR_COL] <= FORECAST_YEAR)]
    ok = []
    for pid, g in d.groupby(ENTITY_COL):
        if (set(g[YEAR_COL]) == need and not g[YEAR_COL].duplicated().any()
                and g[cols].notnull().all().all()):
            ok.append(pid)
    return sorted(ok)


def transform_external(E):
    """(T, n_ext) raw external values -> model inputs, row-aligned with the
    years (row 0 has no previous year; it is never used in a window)."""
    out = np.zeros_like(E, dtype=float)
    for k, col in enumerate(EXTERNAL_VARS):
        x, how = E[:, k], EXTERNAL_TRANSFORM[col]
        if how == "diff":
            out[1:, k] = x[1:] - x[:-1]
        elif how == "logdiff":
            lx = np.log(x)
            out[1:, k] = lx[1:] - lx[:-1]
        elif how == "log1p":
            out[:, k] = np.log1p(x)
        elif how == "level":
            out[:, k] = x
        else:
            raise ValueError(f"unknown transform '{how}' for {col}")
    return out


def build_samples(df, cities):
    """One row per (city, target year t) with t-WINDOW-1 >= FIRST_YEAR.

    D_lags   (n, WINDOW, 6)  lagged deltas of all indicators: t-3, t-2, t-1
    X_ext    (n, WINDOW, E)  growth-type externals at t-3, t-2, t-1
    ent      (n,)            city index (0..n_cities-1)
    y_delta  (n, 6)          delta at t   (the training target)
    y_prev   (n, 6)          level at t-1 (observed)
    y_level  (n, 6)          level at t   (truth for scoring)
    year     (n,)            t
    """
    D_l, X_l, ent, y_delta, y_prev, y_level, year = [], [], [], [], [], [], []
    for idx, pid in enumerate(cities):
        g = df[df[ENTITY_COL] == pid].sort_values(YEAR_COL)
        Y = g[TARGET_VARS].values.astype(float)                  # (T, 6) levels
        D = np.vstack([np.full((1, Y.shape[1]), np.nan), np.diff(Y, axis=0)])
        X = transform_external(g[EXTERNAL_VARS].values.astype(float))
        yrs = g[YEAR_COL].values
        for r in range(WINDOW + 1, len(g)):                      # needs D[r-WINDOW] valid
            D_l.append(D[r - WINDOW:r])
            X_l.append(X[r - WINDOW:r])
            ent.append(idx)
            y_delta.append(D[r]); y_prev.append(Y[r - 1]); y_level.append(Y[r])
            year.append(yrs[r])
    S = dict(D_lags=np.array(D_l, "float32"), X_ext=np.array(X_l, "float32"),
             ent=np.array(ent, "int32"), y_delta=np.array(y_delta, "float32"),
             y_prev=np.array(y_prev, "float64"), y_level=np.array(y_level, "float64"),
             year=np.array(year))
    for k in ("D_lags", "X_ext", "y_delta"):
        assert np.isfinite(S[k]).all(), f"non-finite values in {k}"
    return S


def make_inputs(S, target_idx, use_ext):
    """target_idx=None -> MTL (all six indicators' lagged deltas);
    target_idx=j       -> STL (only indicator j's own lagged delta).
    use_ext=True adds the external variables ("full")."""
    dy = S["D_lags"] if target_idx is None else S["D_lags"][:, :, [target_idx]]
    parts = [dy] + ([S["X_ext"]] if use_ext else [])
    return np.concatenate(parts, axis=2).astype("float32")

# ========================================================= LSTM MODEL =====

def build_lstm(n_features, n_cities, n_out):
    """LSTM over the window + city embedding -> Dense -> n_out deltas."""
    seq_in = Input(shape=(WINDOW, n_features), name="sequence_input")
    ent_in = Input(shape=(1,), name="entity_input")
    lstm_out = LSTM(LSTM_UNITS, activation="tanh")(seq_in)
    emb = Flatten()(Embedding(n_cities, EMBEDDING_DIM, name="entity_embedding")(ent_in))
    hidden = Dense(DENSE_UNITS, activation="relu")(Concatenate()([lstm_out, emb]))
    model = Model([seq_in, ent_in], Dense(n_out, name="delta_out")(hidden))
    model.compile(optimizer="adam", loss="mse")
    return model

# =================================================== XGBOOST / KNN MODELS =

def fit_predict_xgb(X_fit, y_fit, X_val, y_val, X_te):
    """k=1 target -> ordinary booster; k>1 -> ONE vector-leaf booster (MTL)."""
    k = y_fit.shape[1]
    params = dict(XGB_PARAMS)
    if k > 1:
        params["multi_strategy"] = "multi_output_tree"
        yf, yv = y_fit, y_val
    else:
        yf, yv = y_fit.ravel(), y_val.ravel()
    model = xgb.XGBRegressor(n_estimators=XGB_N_ESTIMATORS,
                             early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS, **params)
    model.fit(X_fit, yf, eval_set=[(X_val, yv)], verbose=False)
    return model.predict(X_te).reshape(len(X_te), -1)


def fit_predict_knn(X_fit, y_fit, X_val, y_val, X_te):
    """Pick ONE k by validation MedAE (averaged over the k targets), then
    refit on fit+val. Returns (predictions, chosen k)."""
    scores = []
    for kk in KNN_K_GRID:
        knn = KNeighborsRegressor(n_neighbors=min(kk, len(X_fit)), weights=KNN_WEIGHTS)
        pred = knn.fit(X_fit, y_fit).predict(X_val).reshape(len(X_val), -1)
        scores.append(float(np.median(np.abs(y_val - pred), axis=0).mean()))
    best_k = KNN_K_GRID[int(np.argmin(scores))]
    X_all, y_all = np.vstack([X_fit, X_val]), np.vstack([y_fit, y_val])
    knn = KNeighborsRegressor(n_neighbors=min(best_k, len(X_all)), weights=KNN_WEIGHTS)
    return knn.fit(X_all, y_all).predict(X_te).reshape(len(X_te), -1), best_k

# ======================================== ONE DESIGN -> THREE FAMILIES ====

def fit_predict_design(X, ent, y, year, n_cities, label):
    """Fit LSTM, XGBoost and KNN on one input design.
      X (n, WINDOW, F) inputs | ent (n,) city idx | y (n, k) raw deltas
    Returns {family: (n_test, k) predicted deltas for FORECAST_YEAR}."""
    n_out = y.shape[1]
    fit = year <= TRAIN_END_YEAR - VAL_YEARS
    val = (year <= TRAIN_END_YEAR) & ~fit
    test = year == FORECAST_YEAR

    # --- LSTM (fresh EarlyStopping per fit) ---
    model = build_lstm(X.shape[2], n_cities, n_out)
    stopper = EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    model.fit([X[fit], ent[fit]], y[fit],
              validation_data=([X[val], ent[val]], y[val]),
              epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=[stopper])
    lstm_pred = model.predict([X[test], ent[test]], verbose=0).reshape(int(test.sum()), -1)

    # --- city embeddings harvested from this LSTM, attached to flat windows ---
    emb = model.get_layer("entity_embedding").get_weights()[0]      # (n_cities, dim)

    def flat(mask):
        return np.hstack([X[mask].reshape(int(mask.sum()), -1), emb[ent[mask]]])

    X_fit, X_val, X_te = flat(fit), flat(val), flat(test)
    xgb_pred = fit_predict_xgb(X_fit, y[fit], X_val, y[val], X_te)
    knn_pred, best_k = fit_predict_knn(X_fit, y[fit], X_val, y[val], X_te)
    print(f"    {label}: KNN k = {best_k}")
    return {"LSTM": lstm_pred, "XGB": xgb_pred, "KNN": knn_pred}

# ======================================================= ARIMA BASELINE ===

def arima_forecast(df, cities, target):
    """ARIMA(1,1,0) per city on levels through TRAIN_END_YEAR -> FORECAST_YEAR."""
    out = []
    for pid in cities:
        g = df[(df[ENTITY_COL] == pid) & (df[YEAR_COL] <= TRAIN_END_YEAR)]
        s = g.sort_values(YEAR_COL)[target].astype(float).values
        try:
            pred = float(np.asarray(ARIMA(s, order=(5, 1, 0)).fit().forecast(steps=1))[0])
        except Exception:
            pred = float(s[-1])
        out.append(pred)
    return np.array(out)

# =============================================================== MAIN =====

df = load_data(DATA_PATH)
n_total = df[ENTITY_COL].nunique()
cities = eligible_cities(df, TARGET_VARS + EXTERNAL_VARS)
n_cities = len(cities)
print(f"{n_cities}/{n_total} cities have every year {FIRST_YEAR}-{FORECAST_YEAR} "
      f"with complete target + external data -> used by ALL models.")

df = df[df[ENTITY_COL].isin(cities) &
        (df[YEAR_COL] >= FIRST_YEAR) & (df[YEAR_COL] <= FORECAST_YEAR)].copy()

S = build_samples(df, cities)
year, ent = S["year"], S["ent"]
test = year == FORECAST_YEAR
assert int(test.sum()) == n_cities, "expected exactly one test row per city"
y_true, y_prev = S["y_level"][test], S["y_prev"][test]         # (n_cities, 6), city order
print(f"samples: {int((year <= TRAIN_END_YEAR).sum())} training rows "
      f"(target years {year.min()}-{TRAIN_END_YEAR}), {n_cities} test rows ({FORECAST_YEAR}).")

preds = {}                                                     # model name -> (n_cities, 6) level forecasts

# ---- ARIMA ----
print("\nFitting ARIMA(1,1,0) per city...")
preds["ARIMA"] = np.column_stack([arima_forecast(df, cities, t) for t in TARGET_VARS])

# ---- ML: {internal, full} x {MTL, STL} x {LSTM, XGB, KNN} ----
for feature_label, use_ext in [("internal", False), ("full", True)]:
    print(f"\n'{feature_label}' designs...")

    # MTL: one model, all six indicators jointly
    print("  MTL")
    out = fit_predict_design(make_inputs(S, None, use_ext), ent, S["y_delta"],
                             year, n_cities, f"MTL-{feature_label}")
    for fam in FAMILIES:
        preds[f"MTL-{fam} ({feature_label})"] = y_prev + out[fam]

    # STL: six independent models, own lags only (+ externals if full)
    print("  STL")
    stl = {fam: np.zeros_like(y_prev) for fam in FAMILIES}
    for j, target in enumerate(TARGET_VARS):
        out = fit_predict_design(make_inputs(S, j, use_ext), ent, S["y_delta"][:, [j]],
                                 year, n_cities, f"STL-{feature_label} | {target}")
        for fam in FAMILIES:
            stl[fam][:, j] = y_prev[:, j] + out[fam][:, 0]
    for fam in FAMILIES:
        preds[f"STL-{fam} ({feature_label})"] = stl[fam]

print("\nDone fitting. Building report...")

# ============================================================ REPORT ======
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)

# column order: ARIMA | STL internal | STL full | MTL internal | MTL full
MODEL_ORDER = ["ARIMA",
               "STL-LSTM (internal)", "STL-XGB (internal)", "STL-KNN (internal)",
               "STL-LSTM (full)", "STL-XGB (full)", "STL-KNN (full)",
               "MTL-LSTM (internal)", "MTL-XGB (internal)", "MTL-KNN (internal)",
               "MTL-LSTM (full)", "MTL-XGB (full)", "MTL-KNN (full)"]

HYPOTHESES = [
    ("LSTM1", "MTL-LSTM (full) < ARIMA",               "MTL-LSTM (full)", "ARIMA"),
    ("LSTM2", "MTL-LSTM (full) < STL-LSTM (full)",      "MTL-LSTM (full)", "STL-LSTM (full)"),
    ("LSTM3", "MTL-LSTM (full) < MTL-LSTM (internal)",  "MTL-LSTM (full)", "MTL-LSTM (internal)"),
    ("LSTM4", "STL-LSTM (full) < STL-LSTM (internal)",  "STL-LSTM (full)", "STL-LSTM (internal)"),
    ("XGB1", "MTL-XGB (full) < ARIMA",                  "MTL-XGB (full)", "ARIMA"),
    ("XGB2", "MTL-XGB (full) < STL-XGB (full)",         "MTL-XGB (full)", "STL-XGB (full)"),
    ("XGB3", "MTL-XGB (full) < MTL-XGB (internal)",     "MTL-XGB (full)", "MTL-XGB (internal)"),
    ("XGB4", "STL-XGB (full) < STL-XGB (internal)",     "STL-XGB (full)", "STL-XGB (internal)"),
    ("KNN1", "MTL-KNN (full) < ARIMA",                  "MTL-KNN (full)", "ARIMA"),
    ("KNN2", "MTL-KNN (full) < STL-KNN (full)",         "MTL-KNN (full)", "STL-KNN (full)"),
    ("KNN3", "MTL-KNN (full) < MTL-KNN (internal)",     "MTL-KNN (full)", "MTL-KNN (internal)"),
    ("KNN4", "STL-KNN (full) < STL-KNN (internal)",     "STL-KNN (full)", "STL-KNN (internal)"),
    ("CROSS1", "MTL-XGB (full) < MTL-KNN (full)",       "MTL-XGB (full)", "MTL-KNN (full)"),
    ("CROSS2", "MTL-LSTM (full) < MTL-XGB (full)",      "MTL-LSTM (full)", "MTL-XGB (full)"),
    ("CROSS3", "MTL-LSTM (full) < MTL-KNN (full)",      "MTL-LSTM (full)", "MTL-KNN (full)"),
]


def smape_city(y, yhat):
    denom = np.abs(y) + np.abs(yhat)
    return np.where(denom != 0, 2.0 * np.abs(y - yhat) / np.where(denom != 0, denom, 1.0), 0.0)


def win_rates(E):
    """E: DataFrame (cities x models) of absolute errors. Share (%) of cities
    where each model has the smallest error; exact ties are split equally."""
    a = E.values
    best = np.isclose(a, a.min(axis=1, keepdims=True), rtol=0.0, atol=1e-12)
    w = best / best.sum(axis=1, keepdims=True)
    return pd.Series(w.mean(axis=0) * 100, index=E.columns)


rows, overview_rows, win_by_target = [], [], {}
for j, target in enumerate(TARGET_VARS):
    E = pd.DataFrame({m: np.abs(y_true[:, j] - preds[m][:, j]) for m in MODEL_ORDER})
    SM = pd.DataFrame({m: smape_city(y_true[:, j], preds[m][:, j]) for m in MODEL_ORDER})
    medae, smdape, win = E.median(), SM.median() * 100, win_rates(E)
    win_by_target[target] = win

    print(f"\n=== {target} (lowest MedAE: {medae.idxmin()}) ===")
    disp = pd.DataFrame({"n_cities": pd.Series(n_cities, index=MODEL_ORDER).astype(str),
                         "MedAE": medae.map("{:.4f}".format),
                         "sMdAPE_%": smdape.map("{:.2f}".format)}).T
    print(disp.to_string())

    ov = {"target_variable": target}
    for h_id, label, a, b in HYPOTHESES:
        supported = bool(medae[a] < medae[b])
        print(f"  {h_id} ({label}): {'SUPPORTED' if supported else 'NOT SUPPORTED'} "
              f"[{medae[a]:.4f} vs {medae[b]:.4f}]")
        ov[h_id] = "Y" if supported else "N"
    overview_rows.append(ov)

    for m in MODEL_ORDER:
        rows.append({"target_variable": target, "model": m, "n_cities": n_cities,
                     "MedAE": medae[m], "sMdAPE_%": smdape[m], "win_rate_%": win[m]})

results_df = pd.DataFrame(rows)
overview_df = pd.DataFrame(overview_rows).set_index("target_variable")
print("\n=== Hypothesis overview by median MedAE (Y = supported, N = not supported) ===")
print(overview_df.to_string())

win_tbl = pd.DataFrame(win_by_target).T[MODEL_ORDER]          # rows: targets | columns: models
win_tbl.loc["ALL (mean)"] = win_tbl.mean()
print("\n=== Winning rate (%): share of cities where the model has the smallest "
      "absolute error among all models ===")
print(win_tbl.to_string(float_format=lambda v: f"{v:.1f}"))

print(f"""
Notes
- All models are scored on the same {n_cities} cities, one test year ({FORECAST_YEAR}).
- ML models predict raw deltas (y_t - y_(t-1)) from 3 lagged years and are rebuilt as
  y_(t-1) + delta_pred; no input or target scaling. Externals enter as year-over-year
  level changes, except zero-heavy shocks (see EXTERNAL_TRANSFORM).
- Winning rate: per indicator, each city is "won" by the model with the smallest absolute
  error among all {len(MODEL_ORDER)} models (exact ties split equally); rows sum to 100.
- City errors share the same {FORECAST_YEAR} shocks, so treat differences as descriptive.
""")

259/274 cities have every year 2013-2024 with complete target + external data -> used by ALL models.
samples: 1295 training rows (target years 2019-2023), 259 test rows (2024).

Fitting ARIMA(1,1,0) per city...

'internal' designs...
  MTL
    MTL-internal: KNN k = 30
  STL
    STL-internal | tot_rev_gn: KNN k = 30
    STL-internal | tot_exp_gn: KNN k = 15
    STL-internal | cash_ga: KNN k = 30
    STL-internal | una_ga: KNN k = 20
    STL-internal | una_ba: KNN k = 30
    STL-internal | nic: KNN k = 15
    STL-internal | total_assets_GA: KNN k = 10
    STL-internal | total_liabilities_GA: KNN k = 7
    STL-internal | tot_fnp: KNN k = 30

'full' designs...
  MTL
    MTL-full: KNN k = 30
  STL
    STL-full | tot_rev_gn: KNN k = 30
    STL-full | tot_exp_gn: KNN k = 15
    STL-full | cash_ga: KNN k = 30
    STL-full | una_ga: KNN k = 20
    STL-full | una_ba: KNN k = 30
    STL-full | nic: KNN k = 15
    STL-full | total_assets_GA: KNN k = 10
    STL-full | total_liabilities_GA: KNN k = 